# Abstract

Accurate and early classification of skin lesions is critical for effective clinical diagnosis. This study presents a deep learning–based ensemble framework for automated skin lesion classification. Seven models—CNN, ResNet-50, DenseNet-121, EfficientNet-B3, ConvNeXt-Tiny, MobileNetV3, and Vision Transformer (ViT)—were trained on the HAM10000 dataset and externally validated on ISIC 2019.

Results show that ensemble methods significantly improve performance under domain shift [(Thwin and Park, 2024); (Efat et al., 2024)]. DenseNet-121 and ConvNeXt-Tiny achieved the highest generalization, while ResNet-50 and ViT struggled with external AUC. We also computed Expected Calibration Error (ECE) to assess model reliability and used Grad-CAM visualizations to interpret ensemble predictions across lesion classes.

All outputs, including classification reports, ROC curves, calibration plots, Grad-CAM heatmaps, LaTeX tables, and prediction files, are export-ready. This work provides a reproducible benchmark for clinically relevant, interpretable, and generalizable AI diagnostic systems.

**Keywords:** Skin lesion classification, deep learning, external validation, ISIC 2019, ensemble learning, Grad-CAM, calibration, medical AI.

# 1. Introduction

Skin cancer, particularly melanoma, is among the most prevalent and deadly forms of cancer worldwide. Early detection is essential for improving survival rates, and dermoscopic imaging plays a vital role in the non-invasive assessment of skin lesions. However, manual diagnosis by dermatologists can be subjective, time-consuming, and error-prone—especially in the absence of consistent clinical expertise.

Deep learning approaches, especially Convolutional Neural Networks (CNNs), have demonstrated state-of-the-art performance in automating skin lesion classification. Numerous studies have successfully leveraged CNNs in clinical decision support systems, particularly for tasks involving melanoma detection [(Tschandl et al., 2023)]. More recently, Vision Transformers (ViTs) have emerged as a competitive alternative due to their capacity to model global image context and long-range dependencies [(Zhao et al., 2024)].

However, many existing models are evaluated only on internal test sets, raising concerns about their ability to generalize across diverse imaging conditions, devices, and patient populations. External validation—such as testing on the ISIC 2019 dataset—is essential for assessing a model’s robustness under domain shift.

In this work, we present a robust and generalizable ensemble learning pipeline that combines seven deep architectures—CNN, ResNet-50, DenseNet-121, EfficientNet-B3, ConvNeXt-Tiny, MobileNetV3, and ViT—trained on HAM10000 and externally evaluated on ISIC 2019. Our approach emphasizes both **predictive performance** and **interpretability**, incorporating ensemble Grad-CAM heatmaps and Expected Calibration Error (ECE) analysis to provide clinically meaningful insights. This notebook serves as a reproducible benchmark for building trustworthy AI-assisted diagnostic tools in dermatology.

# 	2.	Materials and Methods

## 2.1 Environment Setup

###  Install Required Packages (Grad-CAM + EfficientNet)

In [ ]:
# Install Grad-CAM 
!pip install -q git+https://github.com/jacobgil/pytorch-grad-cam.git

# Optional: efficientnet_pytorch if needed
!pip install -q efficientnet_pytorch

### Re-import Grad-CAM Modules After Installation

In [ ]:
# Re-import Grad-CAM components after installation
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

print("Grad-CAM successfull|y re-imported.")

### Import Libraries and Define Device

In [ ]:
# Core
import os
import json
import copy
import random
import warnings
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
from matplotlib.patches import Patch

# Imaging
from PIL import Image
import cv2                            # optional, for overlays if available
from skimage.transform import resize  # only if need to resize CAM masks

# Progress
from tqdm import tqdm

# PyTorch
import torch
import torch.nn as nn
from torch.optim import Adam
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

# TorchVision
import torchvision
from torchvision import transforms
from torchvision import models
from torchvision.models import (
    efficientnet_b3, resnet50, densenet121, mobilenet_v3_large,
    convnext_tiny  # already imported before; moved here to consolidate
)
from torchvision.datasets.folder import default_loader

# Hugging Face Transformers (for ViT only)
from transformers import ViTConfig, ViTForImageClassification, ViTImageProcessor

# timm (preinstalled on Kaggle) for ViT and many CNNs

# EfficientNet from efficientnet_pytorch (required for loading EfficientNet models not from timm)

# SciPy / Stats
from scipy.optimize import linear_sum_assignment
from scipy.stats import binomtest

# Scikit-learn
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, roc_curve, 
    confusion_matrix, classification_report
)
from sklearn.metrics import auc as sk_auc  # avoid conflict with built-in auc

# McNemar's test
from statsmodels.stats.contingency_tables import mcnemar

# Optional: Grad-CAM (skip gracefully if not available) 
HAS_GRADCAM = True
try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam import GradCAM as PTGradCAM  
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
except Exception:
    HAS_GRADCAM = False
    print("Grad-CAM not available; CAM visualizations will be skipped.")

import shutil

# Both timm and HuggingFace transformers available for model loading (e.g., ViT)

# Config / Reproducibility
warnings.filterwarnings("ignore")

def seed_all(seed: int = 123):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_all(123)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
sns.set_theme(context="notebook", style="whitegrid")
plt.rcParams["figure.dpi"] = 120

### Global Label Mapping & Class Names

In [ ]:
df = pd.read_csv("/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_metadata.csv")
label_map = {name:i for i,name in enumerate(sorted(df['dx'].unique()))}
class_names = [k for k,_ in sorted(label_map.items(), key=lambda x:x[1])]
df['label'] = df['dx'].map(label_map).astype(int)

### Globals & Paths (Once)

In [ ]:
# Canonical class list must already exist (we have it from 'Global Label Mapping & Class Names')
assert 'class_names' in globals() and isinstance(class_names, (list, tuple)) and len(class_names) > 0, \
    "Define class_names first (canonical order)."
NUM_C = len(class_names)

# Image folders and resolver (used throughout the ensemble cells)
IMG_DIRS = [
    "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1",
    "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2",
]

def find_image_path_from_id(image_id: str):
    fname = f"{image_id}.jpg"
    for d in IMG_DIRS:
        p = os.path.join(d, fname)
        if os.path.exists(p):
            return p
    return None

### Define ROC-AUC Plotting Utility (Global)

In [ ]:
def plot_multiclass_roc(all_labels, all_probs, class_names, title, outfile):
    """
    all_labels: 1D array-like of true integer labels (0..C-1)
    all_probs:  2D array-like of predicted probabilities, shape [N, C]
    class_names: list of class names in the same index order
    """
    y_true = label_binarize(all_labels, classes=list(range(len(class_names))))
    y_prob = np.asarray(all_probs)

    plt.figure(figsize=(10, 8))
    for i, cname in enumerate(class_names):
        fpr, tpr, _ = roc_curve(y_true[:, i], y_prob[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{cname} (AUC = {roc_auc:.2f})")

    plt.plot([0, 1], [0, 1], 'k--', label='Random')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(title)
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(outfile, dpi=300)
    plt.show()

    macro_auc = roc_auc_score(y_true, y_prob, average='macro')
    micro_auc = roc_auc_score(y_true, y_prob, average='micro')
    print(f"Macro AUC: {macro_auc:.4f}")
    print(f"Micro AUC: {micro_auc:.4f}")

## 2.2 Dataset Preparation

### Custom Dataset Class for HAM10000

In [ ]:
class HAM10000Dataset(Dataset):
    def __init__(self, dataframe, image_dirs, transform=None):
        self.data = dataframe
        self.image_dirs = image_dirs
        self.transform = transform
        self.loader = default_loader

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        filename = row['image_id'] + ".jpg"

        # Search for image in both directories
        for d in self.image_dirs:
            img_path = os.path.join(d, filename)
            if os.path.exists(img_path):
                break
        else:
            raise FileNotFoundError(f"Image {filename} not found in provided directories.")

        image = self.loader(img_path).convert("RGB")
        label = torch.tensor(int(row['label']), dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        return image, label

### Load and Prepare HAM10000 Test Dataset

In [ ]:
# Point to both image folders
image_dirs = [
    "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1",
    "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2",
]

# Load metadata
df = pd.read_csv("/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_metadata.csv")

# Map dx -> integer label (keep a stable, sorted mapping)
label_names = sorted(df["dx"].unique())
label_map = {name: i for i, name in enumerate(label_names)}
df["label"] = df["dx"].map(label_map).astype(int)

# Keep only rows whose image file actually exists in either folder
def exists_in_dirs(img_id: str) -> bool:
    fname = f"{img_id}.jpg"
    return any(os.path.exists(os.path.join(d, fname)) for d in image_dirs)

df = df[df["image_id"].apply(exists_in_dirs)].reset_index(drop=True)

# Create a previewable path column (not used by HAM10000Dataset, but handy for debugging)
def make_path(img_id: str) -> str:
    fname = f"{img_id}.jpg"
    p1 = os.path.join(image_dirs[0], fname)
    p2 = os.path.join(image_dirs[1], fname)
    return p1 if os.path.exists(p1) else p2
df["image_path"] = df["image_id"].apply(make_path)

# Stratified splits: 70% train / 15% val / 15% test
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print("Split sizes:", len(train_df), len(val_df), len(test_df))
print("Label map:", label_map)

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

# Datasets (use HAM10000Dataset class that searches both folders)
train_dataset = HAM10000Dataset(train_df, image_dirs=image_dirs, transform=train_transform)
val_dataset   = HAM10000Dataset(val_df,   image_dirs=image_dirs, transform=val_transform)
test_dataset  = HAM10000Dataset(test_df,  image_dirs=image_dirs, transform=val_transform)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# Quick sanity check
for imgs, labels in train_loader:
    print("Sample batch:", imgs.shape, labels.shape)
    break

### Utility: Visualize Per Class Gradcam (PTGradCAM)

In [ ]:
def visualize_per_class_gradcam(model, dataset, class_names, target_layer):
    model.eval()
    seen_classes = set()

    # Convert layer name to actual layer
    target_module = dict(model.named_modules())[target_layer]
    cam = PTGradCAM(model=model, target_layers=[target_module])

    for i in range(len(dataset)):
        image, label = dataset[i]
        if label in seen_classes:
            continue

        img_in = image.unsqueeze(0).to(device)
        with torch.no_grad():
            pred = model(img_in).argmax(1).item()

        if pred == label:
            img_np = image.permute(1, 2, 0).cpu().numpy()
            img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)

            grayscale_cam = cam(img_in, targets=[ClassifierOutputTarget(label)])[0]
            overlay = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)

            plt.figure(figsize=(6, 3))
            plt.suptitle(f"Class: {class_names[label]}")
            plt.subplot(1, 2, 1)
            plt.imshow(img_np)
            plt.title("Original")
            plt.axis('off')

            plt.subplot(1, 2, 2)
            plt.imshow(overlay)
            plt.title("Grad-CAM")
            plt.axis('off')
            plt.show()

            seen_classes.add(label)
            if len(seen_classes) == len(class_names):
                break

    try:
        cam.activations_and_grads.release()
    except:
        pass
    del cam

## 2.3 Model Architectures

## CNN (Baseline Model)

In [ ]:
class CNNModel(nn.Module):
    def __init__(self, num_classes=7):
        super(CNNModel, self).__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        # 224x224 -> after 4 pools -> 14x14 feature map with 256 channels
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

### CNN Training Function with Early Stopping (50 Epochs)

In [ ]:
# CNN Training Function with Early Stopping
def train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=50, patience=7):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    epochs_no_improve = 0

    model.to(device)

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 30)

        # Train
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        for images, labels in tqdm(train_loader, desc="Training", leave=False):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)
            train_correct += (outputs.argmax(1) == labels).sum().item()
            train_total += labels.size(0)

        train_loss /= train_total
        train_acc = train_correct / train_total

        # Validate
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc="Validating", leave=False):
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_total += labels.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total

        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"Val   Loss: {val_loss:.4f}, Val   Acc: {val_acc:.4f}")

        # Early stopping + best checkpoint
        if val_acc > best_acc:
            best_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(best_model_wts, "/kaggle/working/cnn_ham10000_best_model.pth")
            print("Best CNN model saved.")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"\nEarly stopping triggered at epoch {epoch+1}")
                break

    return model, best_model_wts

### CNN Training: 50 Epochs with Early Stopping

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNNModel(num_classes=7).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=1e-4)

trained_model, best_model_wts = train_model(
    model, train_loader, val_loader, criterion, optimizer,
    device, num_epochs=50, patience=7
)

# Save best CNN weights
model.load_state_dict(best_model_wts)
torch.save(model.state_dict(), "/kaggle/working/cnn_ham10000_best_model.pth")
print("CNN model saved to /kaggle/working/cnn_ham10000_best_model.pth")

### Reload the Model

In [ ]:
cnn_model = CNNModel(num_classes=7)
cnn_model.load_state_dict(torch.load("/kaggle/input/kaggleinputham10000-model-weights/cnn_ham10000_best_model.pth", map_location=device))
cnn_model = cnn_model.to(device)
cnn_model.eval()

### Evaluate on Test Set — Metrics & Confusion Matrix

In [ ]:
all_preds = []
all_labels = []
all_probs = []

cnn_model.eval()
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images, labels = images.to(device), labels.to(device)
        outputs = cnn_model(images)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(probs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# Convert to NumPy arrays
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

# Accuracy, F1 score
acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average="weighted")

# ROC AUC (one-vs-rest)
try:
    auc_score = roc_auc_score(all_labels, all_probs, multi_class="ovr")
except:
    auc_score = "N/A (AUC needs >1 class in test set)"

# Print metrics
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=class_names))
print(f"Test Accuracy: {acc:.4f}")
print(f"Weighted F1 Score: {f1:.4f}")
print(f"ROC AUC Score: {auc_score}")

# Confusion Matrix Plot
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix.png", dpi=300)
plt.show()

print("Confusion matrix saved to 'confusion_matrix.png'.")

### Plot ROC Curves for Multiclass Classification (Per Class)

In [ ]:
def plot_multiclass_roc(all_labels, all_probs, class_names, title, outfile):

    y_true = label_binarize(all_labels, classes=list(range(len(class_names))))
    y_prob = all_probs

    plt.figure(figsize=(12, 10))

    for i, cname in enumerate(class_names):
        fpr, tpr, _ = roc_curve(y_true[:, i], y_prob[:, i])
        roc_auc = sk_auc(fpr, tpr)  # <<< fix: use sk_auc instead of auc
        plt.plot(fpr, tpr, label=f"{cname} (AUC = {roc_auc:.2f})")

    plt.plot([0, 1], [0, 1], "k--", label="Random")
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(outfile, dpi=300)
    plt.show()

### ROC‑AUC Curves

In [ ]:
plot_multiclass_roc(
    all_labels=all_labels,
    all_probs=all_probs,
    class_names=class_names,
    title="ROC Curves — CNN",
    outfile="/kaggle/working/roc_curves_cnn.png"
)

### Grad-CAM Generation for Test Images — CNN (Plot + Save, with Fallback)

In [ ]:
# Sanity checks
assert 'cnn_model' in globals(), "cnn_model not found. Reload the trained CNN first."
assert 'test_loader' in globals(), "test_loader not found."
assert 'class_names' in globals(), "class_names not found."
assert 'device' in globals(), "device not set."

# Helpers
def inv_norm_05(img_t):
    """Invert Normalize([0.5]*3,[0.5]*3) -> RGB [0,1]. img_t: CHW tensor."""
    x = img_t.detach().cpu().clone()
    for c in range(3):
        x[c] = x[c] * 0.5 + 0.5
    return x.clamp(0, 1).permute(1, 2, 0).numpy()

def find_last_conv_module(net: nn.Module):
    last = None
    for m in net.modules():
        if isinstance(m, nn.Conv2d):
            last = m
    if last is None:
        raise RuntimeError("No Conv2d layer found in the CNN.")
    return last

# Setup Grad‑CAM
cnn_model.eval()
target_layer = find_last_conv_module(cnn_model)
cam = PTGradCAM(model=cnn_model, target_layers=[target_layer])

got = {c: False for c in class_names}
cam_results = {}

# Pass 1: prefer correctly predicted examples
for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
    with torch.no_grad():
        preds = cnn_model(images).argmax(1)

    for i in range(images.size(0)):
        t_lbl = labels[i].item()
        p_lbl = preds[i].item()
        cname = class_names[t_lbl]
        if got[cname] or p_lbl != t_lbl:
            continue

        it = images[i].unsqueeze(0).requires_grad_(True)
        cnn_model.zero_grad()
        gcam = cam(input_tensor=it, targets=[ClassifierOutputTarget(p_lbl)])[0]

        rgb01 = inv_norm_05(images[i])
        cam_img = show_cam_on_image(rgb01, gcam, use_rgb=True)

        cam_results[cname] = (rgb01, cam_img)  # store original (denorm) + overlay
        got[cname] = True

    if all(got.values()):
        break

# Pass 2: fallback for missing classes (use first available, even if misclassified)
if not all(got.values()):
    missing = [c for c, v in got.items() if not v]
    print(f"[Info] No correct predictions for classes: {missing}. Falling back to first available sample.")
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        for i in range(images.size(0)):
            t_lbl = labels[i].item()
            cname = class_names[t_lbl]
            if got[cname]:
                continue

            it = images[i].unsqueeze(0).requires_grad_(True)
            cnn_model.zero_grad()
            # drive CAM toward the true label for visualization
            gcam = cam(input_tensor=it, targets=[ClassifierOutputTarget(t_lbl)])[0]

            rgb01 = inv_norm_05(images[i])
            cam_img = show_cam_on_image(rgb01, gcam, use_rgb=True)

            cam_results[cname] = (rgb01, cam_img)
            got[cname] = True

        if all(got.values()):
            break

# Plot & save
out_dir = Path("/kaggle/working/gradcam_cnn_per_class")
out_dir.mkdir(parents=True, exist_ok=True)

if not cam_results:
    print("[Warning] No Grad‑CAM images generated for CNN. Check model/loader.")
else:
    for cname, (orig_rgb01, cam_img) in cam_results.items():
        plt.figure(figsize=(8, 4))
        plt.subplot(1, 2, 1); plt.imshow(orig_rgb01); plt.title(f"Original — {cname}"); plt.axis('off')
        plt.subplot(1, 2, 2); plt.imshow(cam_img);    plt.title("Grad‑CAM");          plt.axis('off')
        plt.tight_layout()
        save_path = out_dir / f"gradcam_{cname}.png"
        plt.savefig(save_path, dpi=300, bbox_inches="tight", pad_inches=0.05)
        plt.show()
    print(f"Saved CNN Grad‑CAM images to: {out_dir.resolve()}")

# Cleanup hooks
try:
    cam.activations_and_grads.release()
except Exception:
    pass
del cam

## ResNet-50

### Custom Dataset Loader

In [ ]:
class SkinLesionDataset(Dataset):
    def __init__(self, df, img_dirs, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dirs = img_dirs
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.iloc[idx]['image_id']
        img_name = img_id + ".jpg"

        for img_dir in self.img_dirs:
            img_path = os.path.join(img_dir, img_name)
            if os.path.exists(img_path):
                image = Image.open(img_path).convert("RGB")
                break
        else:
            raise FileNotFoundError(f"{img_name} not found in {self.img_dirs}")

        # Ensure numeric label -> torch.long tensor
        label = int(self.df.iloc[idx]['label'])
        label = torch.tensor(label, dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        return image, label

### Data Preparation

In [ ]:
# Metadata
df = pd.read_csv("/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_metadata.csv")

# Label encoding
label_dict = {label: idx for idx, label in enumerate(df['dx'].unique())}
df['label'] = df['dx'].map(label_dict)
df['image_id'] = df['image_id'].astype(str)

# Train/Val/Test split
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.1, stratify=train_df['label'], random_state=42)

# Reset index to prevent DataLoader bugs
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

### Transforms

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

### Dataset and DataLoader Setup

In [ ]:
img_dirs = [
    "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1",
    "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2"
]

train_dataset = SkinLesionDataset(train_df, img_dirs, transform=train_transform)
val_dataset   = SkinLesionDataset(val_df, img_dirs, transform=test_transform)
test_dataset  = SkinLesionDataset(test_df, img_dirs, transform=test_transform)

### Handle Class Imbalance

In [ ]:
class_counts = train_df['label'].value_counts().sort_index().values
weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
sample_weights = weights[train_df['label'].values]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

### DataLoader Setup

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

### Define ResNet50 Architecture

In [ ]:
# Define ResNet50 model
def get_resnet50_model(num_classes=7):
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)  # load pretrained ResNet50
    for param in model.parameters():
        param.requires_grad = False  # freeze all layers

    # Replace the fully connected layer
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    return model

### Prepare Model, Loss, Optimizer

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

resnet_model = get_resnet50_model(num_classes=7)
resnet_model = resnet_model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet_model.fc.parameters(), lr=1e-4)

### Define Training Function (with Early Stopping)

In [ ]:
def train_model_resnet50(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=50, patience=5):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 50)

        # Training
        model.train()
        running_loss, correct_train, total_train = 0.0, 0, 0
        for images, labels in tqdm(train_loader, desc="Training", leave=False):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss  += loss.item() * images.size(0)
            correct_train += (outputs.argmax(1) == labels).sum().item()
            total_train   += labels.size(0)

        train_loss = running_loss / total_train
        train_acc  = correct_train / total_train

        # Validation
        model.eval()
        val_running, correct_val, total_val = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc="Validation", leave=False):
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss    = criterion(outputs, labels)

                val_running += loss.item() * images.size(0)
                correct_val += (outputs.argmax(1) == labels).sum().item()
                total_val   += labels.size(0)

        val_loss = val_running / total_val
        val_acc  = correct_val / total_val

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

        # Early stopping logic
        if val_acc > best_acc:
            best_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            print("Best ResNet50 model updated.")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered.")
                break

    return model, best_model_wts

### Train the Model

In [ ]:
resnet_model, best_model_wts = train_model_resnet50(
    model=resnet_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=50,
    patience=5
)

# Save best model weights right after training finishes
resnet_model.load_state_dict(best_model_wts)
torch.save(resnet_model.state_dict(), "/kaggle/working/resnet50_ham10000.pth")
print("ResNet50 model saved to /kaggle/working/resnet50_ham10000.pth")

### Reload the Model

In [ ]:
resnet_model = get_resnet50_model(num_classes=7)  # same function used earlier
resnet_model.load_state_dict(
    torch.load("/kaggle/input/resnet50-ham10000-model-weights/resnet50_ham10000.pth", map_location=device)
)
resnet_model = resnet_model.to(device)
resnet_model.eval()

print("ResNet-50 model reloaded from /kaggle/working/resnet50_ham10000.pth")

### Evaluate on Test Set — Metrics & Confusion Matrix

In [ ]:
resnet_model.eval()

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images, labels = images.to(device), labels.to(device)
        outputs = resnet_model(images)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(probs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# Convert to NumPy arrays
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

# Metrics
acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average="weighted")

# ROC AUC (one-vs-rest)
try:
    auc_score = roc_auc_score(all_labels, all_probs, multi_class="ovr")
except ValueError:
    auc_score = "N/A (AUC needs >1 class in test set)"

# Report
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=class_names))
print(f"Test Accuracy: {acc:.4f}")
print(f"Weighted F1 Score: {f1:.4f}")
print(f"ROC AUC Score: {auc_score}")

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix — ResNet-50")
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix_resnet50.png", dpi=300)
plt.show()

print("Confusion matrix saved to 'confusion_matrix_resnet50.png'.")

### ROC‑AUC Curves

In [ ]:
plot_multiclass_roc(
    all_labels=all_labels,
    all_probs=all_probs,
    class_names=class_names,
    title="ROC Curves — ResNet-50",
    outfile="/kaggle/working/roc_curves_resnet50.png"
)

### Grad‑CAM Generation for Test Images — ResNet‑50 (Plot + Save, with Fallback)

In [ ]:
# Sanity checks
assert 'resnet_model' in globals(), "resnet_model not found. Reload/train it first."
assert 'test_loader'   in globals(), "test_loader not found."
assert 'class_names'   in globals(), "class_names not found."
assert 'device'        in globals(), "device not set (cuda/cpu)."

# Helpers (invert ImageNet normalization for correct colors)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def inv_imagenet_norm(img_t):
    """Invert Normalize(IMAGENET_MEAN, IMAGENET_STD) -> RGB [0,1]. img_t: CHW tensor."""
    x = img_t.detach().cpu().clone()
    for c, m, s in zip(range(3), IMAGENET_MEAN, IMAGENET_STD):
        x[c] = x[c] * s + m
    return x.clamp(0, 1).permute(1, 2, 0).numpy()

def find_last_conv_module(net: nn.Module):
    last = None
    for m in net.modules():
        if isinstance(m, nn.Conv2d):
            last = m
    if last is None:
        raise RuntimeError("No Conv2d layer found in the model.")
    return last

# Setup Grad‑CAM target layer (best for ResNet‑50: last bottleneck conv)
resnet_model.eval()
target_layer = getattr(resnet_model.layer4[-1], "conv3", None)
if target_layer is None:  # fallback in case of different variant
    target_layer = find_last_conv_module(resnet_model)

cam = PTGradCAM(model=resnet_model, target_layers=[target_layer])

# Collect one image per class (prefer correctly predicted), else fallback
got = {c: False for c in class_names}
cam_results = {}

# Pass 1 — prefer correct examples
for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
    with torch.no_grad():
        preds = resnet_model(images).argmax(1)

    for i in range(images.size(0)):
        t_lbl = labels[i].item()
        p_lbl = preds[i].item()
        cname = class_names[t_lbl]
        if got[cname] or p_lbl != t_lbl:
            continue

        it = images[i].unsqueeze(0).requires_grad_(True)
        resnet_model.zero_grad()
        gcam = cam(input_tensor=it, targets=[ClassifierOutputTarget(p_lbl)])[0]

        rgb01 = inv_imagenet_norm(images[i])
        cam_img = show_cam_on_image(rgb01, gcam, use_rgb=True)

        cam_results[cname] = (rgb01, cam_img)
        got[cname] = True
    if all(got.values()):
        break

# Pass 2 — fallback: take first available even if misclassified
if not all(got.values()):
    missing = [c for c, v in got.items() if not v]
    print(f"[Info] No correct predictions for classes: {missing}. Falling back to first available sample.")
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        for i in range(images.size(0)):
            t_lbl = labels[i].item()
            cname = class_names[t_lbl]
            if got[cname]:
                continue

            it = images[i].unsqueeze(0).requires_grad_(True)
            resnet_model.zero_grad()
            # drive CAM toward the TRUE label for visualization
            gcam = cam(input_tensor=it, targets=[ClassifierOutputTarget(t_lbl)])[0]

            rgb01 = inv_imagenet_norm(images[i])
            cam_img = show_cam_on_image(rgb01, gcam, use_rgb=True)

            cam_results[cname] = (rgb01, cam_img)
            got[cname] = True
        if all(got.values()):
            break

# Plot & save
out_dir = Path("/kaggle/working/gradcam_resnet50_per_class")
out_dir.mkdir(parents=True, exist_ok=True)

if not cam_results:
    print("[Warning] No Grad‑CAM images generated for ResNet‑50. Check model/loader.")
else:
    for cname, (orig_rgb01, cam_img) in cam_results.items():
        plt.figure(figsize=(8, 4))
        plt.subplot(1, 2, 1); plt.imshow(orig_rgb01); plt.title(f"Original — {cname}"); plt.axis('off')
        plt.subplot(1, 2, 2); plt.imshow(cam_img);   plt.title("Grad‑CAM");          plt.axis('off')
        plt.tight_layout()
        sp = out_dir / f"gradcam_{cname}.png"
        plt.savefig(sp, dpi=300, bbox_inches="tight", pad_inches=0.05)
        plt.show()
    print(f"Saved ResNet‑50 Grad‑CAM images to: {out_dir.resolve()}")

# Cleanup hooks
try:
    cam.activations_and_grads.release()
except Exception:
    pass
del cam

## Vision Transformer (ViT)

### Load ViT Processor

In [ ]:
# Load image processor for ViT
processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")

### Define Custom Dataset Class

In [ ]:
class HAM10000ViTDataset(Dataset):
    def __init__(self, dataframe, image_dirs, processor, transform=None):
        """
        Args:
            dataframe: Pandas DataFrame with columns ['image_id', 'label'].
            image_dirs: List of folders where HAM10000 images are stored.
            processor: HuggingFace ViTImageProcessor.
            transform: Optional torchvision transform (applied before processor).
        """
        self.df = dataframe
        self.image_dirs = image_dirs
        self.processor = processor
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_filename = row["image_id"] + ".jpg"

        # Find the image in one of the directories
        img_path = None
        for folder in self.image_dirs:
            candidate = os.path.join(folder, img_filename)
            if os.path.exists(candidate):
                img_path = candidate
                break

        if img_path is None:
            raise FileNotFoundError(f"Image {img_filename} not found in given folders.")

        # Load image
        image = Image.open(img_path).convert("RGB")

        # Optional torchvision transforms (e.g., augmentation)
        if self.transform:
            image = self.transform(image)

        # Processor: resize, normalize -> pixel_values
        encodings = self.processor(image, return_tensors="pt")

        return {
            "pixel_values": encodings["pixel_values"].squeeze(0),
            "labels": torch.tensor(row["label"], dtype=torch.long)
        }

### Create ViT-Compatible Datasets and Dataloaders

In [ ]:
# Define where the image folders are
image_dirs = [
    "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1",
    "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2"
]

# Create datasets
train_dataset = HAM10000ViTDataset(train_df, image_dirs, processor)
val_dataset   = HAM10000ViTDataset(val_df,   image_dirs, processor)
test_dataset  = HAM10000ViTDataset(test_df,  image_dirs, processor)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False)

### Define & Initialize ViT Model

In [ ]:
vit_model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=len(class_names),
    label2id={name: i for i, name in enumerate(class_names)},
    id2label={i: name for i, name in enumerate(class_names)},
    ignore_mismatched_sizes=True
).to(device)

# Load fine-tuned weights (use weights_only=True for speed & safety)
vit_model.load_state_dict(torch.load(
    "/kaggle/input/vit-ham10000-model-weights/vit_ham10000_best_model.pth",
    map_location=device,
    weights_only=True
))

vit_model.eval()
print("ViT model initialized and fine-tuned weights loaded successfully")

### Train ViT — Early Stopping + Save Best

In [ ]:
# ViT training (dict-based DataLoader: batch["pixel_values"], batch["labels"])
criterion   = nn.CrossEntropyLoss()
optimizer   = Adam(vit_model.parameters(), lr=2e-5)
num_epochs  = 50
patience    = 5

best_val_acc       = 0.0
epochs_no_improve  = 0
best_model_wts     = deepcopy(vit_model.state_dict())  # from copy import deepcopy
vit_model.to(device)

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 30)

    # Train 
    vit_model.train()
    train_loss, train_correct, train_n = 0.0, 0, 0

    for batch in tqdm(train_loader, desc="Training", leave=False):
        inputs = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = vit_model(inputs).logits
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        bs = labels.size(0)
        train_loss   += loss.item() * bs
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_n      += bs

    train_loss /= max(1, train_n)
    train_acc   = train_correct / max(1, train_n)

    # Validate 
    vit_model.eval()
    val_loss, val_correct, val_n = 0.0, 0, 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validating", leave=False):
            inputs = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            outputs = vit_model(inputs).logits
            loss    = criterion(outputs, labels)

            bs = labels.size(0)
            val_loss   += loss.item() * bs
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_n      += bs

    val_loss /= max(1, val_n)
    val_acc   = val_correct / max(1, val_n)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    # Early stopping on Val Acc 
    if val_acc > best_val_acc:
        best_val_acc      = val_acc
        best_model_wts    = deepcopy(vit_model.state_dict())
        epochs_no_improve = 0
        print("Best ViT weights updated.")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

# Restore best and save 
vit_model.load_state_dict(best_model_wts)
out_path = "/kaggle/working/vit_ham10000_best_model.pth"
torch.save(vit_model.state_dict(), out_path)
print(f"Saved: {out_path} | Best Val Acc: {best_val_acc:.4f}")

### Save Fine-tuned ViT Model & Processor in Hugging Face Format (with Reload Test)

In [ ]:
# Ensure the model has the best weights loaded
vit_model.load_state_dict(
    torch.load("/kaggle/input/vit-ham10000-model-weights/vit_ham10000_best_model.pth", map_location="cpu")
)
vit_model.eval()

# Save both model and processor in Hugging Face format
save_dir = "/kaggle/working/vit_ham10000_finetuned"
vit_model.save_pretrained(save_dir)
processor.save_pretrained(save_dir)

print(f"ViT model and processor saved to {save_dir}")

# Example: reload later
vit_model = ViTForImageClassification.from_pretrained(save_dir).to(device)
processor = ViTImageProcessor.from_pretrained(save_dir)
vit_model.eval()

print("Reloaded ViT model and processor successfully.")

### Load Fine-tuned ViT and Evaluate on Test Set

In [ ]:
# Reload processor
processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")

# Recreate architecture with correct num_labels
vit_model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=len(class_names),
    ignore_mismatched_sizes=True
).to(device)

# Load fine-tuned weights
vit_model.load_state_dict(torch.load("/kaggle/input/vit-ham10000-model-weights/vit_ham10000_best_model.pth", map_location=device))
vit_model.eval()

# Evaluation
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        logits = vit_model(pixel_values).logits
        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# Metrics
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average="weighted")
try:
    auc_score = roc_auc_score(all_labels, all_probs, multi_class="ovr")
except ValueError:
    auc_score = "N/A (AUC needs >1 class in test set)"

print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=class_names))
print(f"Test Accuracy: {acc:.4f}")
print(f"Weighted F1 Score: {f1:.4f}")
print(f"ROC AUC Score: {auc_score}")

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix — ViT")
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix_vit.png", dpi=300)
plt.show()

### ViT Grad-CAM Wrapper & Reshape Transform

In [ ]:
class ViTWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.model.eval()
    def forward(self, x):
        return self.model(x).logits

def vit_reshape_transform(tensor, height=14, width=14):
    # remove class token, then B x (H*W) x C -> B x C x H x W
    tensor = tensor[:, 1:, :]
    return tensor.reshape(tensor.size(0), height, width, -1).permute(0, 3, 1, 2)

### ROC‑AUC Curves - ViT

In [ ]:
plot_multiclass_roc(
    all_labels=np.array(all_labels),
    all_probs=np.array(all_probs),
    class_names=class_names,
    title="ROC Curves — ViT",
    outfile="/kaggle/working/roc_curves_vit.png"
)

### Grad‑CAM Generation for Test Images — ViT (Plot + Save, with Fallback)

In [ ]:
# Sanity checks / required objects
assert 'vit_model'   in globals(), "vit_model not found. Train or reload ViT first."
assert 'test_df'     in globals(), "test_df not found. Please define the test set split first."
assert 'class_names' in globals(), "class_names not found. Define from label_map."
assert 'device'      in globals(), "device not set (cuda/cpu)."

# Use the same processor used during training
vit_processor = globals().get('vit_processor', None)
if vit_processor is None:
    from transformers import ViTImageProcessor
    vit_processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")

# Minimal wrappers
if 'ViTWrapper' not in globals():
    class ViTWrapper(nn.Module):
        def __init__(self, model):
            super().__init__()
            self.model = model
            self.model.eval()
        def forward(self, x):
            return self.model(x).logits

if 'vit_reshape_transform' not in globals():
    def vit_reshape_transform(tensor, height=14, width=14):
        tensor = tensor[:, 1:, :]
        return tensor.reshape(tensor.size(0), height, width, -1).permute(0, 3, 1, 2)

# Helpers
def _load_rgb01(path):
    return np.asarray(Image.open(path).convert("RGB")).astype(np.float32) / 255.0

def _row_to_path(row):
    image_id = row["image_id"]
    for d in image_dirs:
        candidate = os.path.join(d, image_id + ".jpg")
        if os.path.exists(candidate):
            return candidate
    return None  # Not found in any directory

# Prepare model + target layer
vit_model.eval()
wrapped_vit = ViTWrapper(vit_model)
try:
    target_layers = [vit_model.vit.encoder.layer[-1].output]
except:
    target_layers = [list(vit_model.vit.encoder.layer)[-1].output]

got = {c: False for c in class_names}
cam_results = {}

# Pass 1 — prefer correct predictions
with GradCAM(model=wrapped_vit, target_layers=target_layers, reshape_transform=vit_reshape_transform) as cam:
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="ViT Grad-CAM (correct pass)"):
        lbl = int(row["label"])
        cname = class_names[lbl]
        if got[cname]:
            continue

        p = _row_to_path(row)
        if p is None or not os.path.exists(p):
            continue

        pil_img = Image.open(p).convert("RGB")
        enc = vit_processor(images=pil_img, return_tensors="pt")
        x = enc["pixel_values"].to(device)

        with torch.no_grad():
            pred = vit_model(x).logits.argmax(1).item()
        if pred != lbl:
            continue

        rgb01 = _load_rgb01(p)
        grayscale_cam = cam(input_tensor=x, targets=[ClassifierOutputTarget(lbl)])[0]

        # Resize original to Grad-CAM mask size
        rgb01_resized = resize(rgb01, (grayscale_cam.shape[0], grayscale_cam.shape[1]), preserve_range=True)

        cam_img = show_cam_on_image(rgb01_resized.astype(np.float32), grayscale_cam, use_rgb=True)
        cam_results[cname] = (rgb01_resized, cam_img)
        got[cname] = True

        if all(got.values()):
            break

# Pass 2 — fallback if some classes missing
if not all(got.values()):
    missing = [c for c, v in got.items() if not v]
    print(f"[Info] No correct predictions for classes: {missing}. Falling back to first available sample.")
    with GradCAM(model=wrapped_vit, target_layers=target_layers, reshape_transform=vit_reshape_transform) as cam:
        for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="ViT Grad-CAM (fallback pass)"):
            lbl = int(row["label"])
            cname = class_names[lbl]
            if got[cname]:
                continue

            p = _row_to_path(row)
            if p is None or not os.path.exists(p):
                continue

            pil_img = Image.open(p).convert("RGB")
            enc = vit_processor(images=pil_img, return_tensors="pt")
            x = enc["pixel_values"].to(device)

            rgb01 = _load_rgb01(p)
            grayscale_cam = cam(input_tensor=x, targets=[ClassifierOutputTarget(lbl)])[0]

            rgb01_resized = resize(rgb01, (grayscale_cam.shape[0], grayscale_cam.shape[1]), preserve_range=True)

            cam_img = show_cam_on_image(rgb01_resized.astype(np.float32), grayscale_cam, use_rgb=True)
            cam_results[cname] = (rgb01_resized, cam_img)
            got[cname] = True

            if all(got.values()):
                break

# Plot & save
out_dir = Path("/kaggle/working/gradcam_vit_per_class")
out_dir.mkdir(parents=True, exist_ok=True)

if not cam_results:
    print("[Warning] No Grad-CAM images generated for ViT. Check model/paths.")
else:
    for cname, (orig_rgb01, cam_img) in cam_results.items():
        plt.figure(figsize=(8, 4))
        plt.subplot(1, 2, 1); plt.imshow(orig_rgb01);  plt.title(f"Original — {cname}"); plt.axis('off')
        plt.subplot(1, 2, 2); plt.imshow(cam_img);     plt.title("Grad-CAM");           plt.axis('off')
        plt.tight_layout()
        save_path = out_dir / f"gradcam_{cname}.png"
        plt.savefig(save_path, dpi=300, bbox_inches="tight", pad_inches=0.05)
        plt.show()
    print(f"Saved ViT Grad-CAM images to: {out_dir.resolve()}")

# Cleanup
try:
    cam.activations_and_grads.release()
except:
    pass
del cam

## DenseNet-121

### Load Metadata, Resolve Image Paths, and Prepare Train/Val/Test

In [ ]:
# Load metadata
df = pd.read_csv("/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_metadata.csv")

# Always (re)create label_map and class_names from the current df
label_map = {name: i for i, name in enumerate(sorted(df["dx"].dropna().unique()))}
class_names = [k for k, _ in sorted(label_map.items(), key=lambda x: x[1])]

# Resolve image paths
image_dir_1 = "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1/"
image_dir_2 = "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2/"

def resolve_path(img_id):
    p1 = os.path.join(image_dir_1, f"{img_id}.jpg")
    p2 = os.path.join(image_dir_2, f"{img_id}.jpg")
    if os.path.exists(p1): 
        return p1
    if os.path.exists(p2): 
        return p2
    return None  # return None if file not found

# Add paths to dataframe
df["path"] = df["image_id"].apply(resolve_path)
df = df.dropna(subset=["path"]).reset_index(drop=True)

# Map dx to integer labels, drop rows if mapping fails
df["label"] = df["dx"].map(label_map)
df = df.dropna(subset=["label"]).reset_index(drop=True)
df["label"] = df["label"].astype(int)

# Split dataset (80/16/20)
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label"],
    random_state=42
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.20,
    stratify=train_df["label"],
    random_state=42
)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print("Classes:", class_names)

### Transforms - DenseNet

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

### Custom Dataset Class for HAM10000 Skin Lesion Images

In [ ]:
class SkinLesionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        label = row["label"]

        if self.transform:
            img = self.transform(img)
        return img, label

### DataLoaders (DenseNet)

In [ ]:
train_dataset = SkinLesionDataset(train_df, transform=train_transform)
val_dataset   = SkinLesionDataset(val_df,   transform=val_transform)
test_dataset  = SkinLesionDataset(test_df,  transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

### Define DenseNet121 Model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Modern API: weights=... instead of pretrained=True
densenet = densenet121(weights=DenseNet121_Weights.DEFAULT)
densenet.classifier = nn.Linear(densenet.classifier.in_features, len(class_names))
densenet = densenet.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = Adam(densenet.parameters(), lr=1e-4)

### Train Model with Early Stopping (FIXED to return best_model_wts)

In [ ]:
def train_model(model, criterion, optimizer, train_loader, val_loader, device, num_epochs=50, patience=5):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    patience_counter = 0

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        model.train()
        running_loss, running_corrects = 0.0, 0

        for inputs, labels in tqdm(train_loader, desc="Training", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            preds = outputs.argmax(1)
            running_loss    += loss.item() * inputs.size(0)
            running_corrects += (preds == labels).sum().item()

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc  = running_corrects / len(train_loader.dataset)
        print(f"Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f}")

        # Validation
        model.eval()
        val_corrects = 0
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc="Validating", leave=False):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                preds = outputs.argmax(1)
                val_corrects += (preds == labels).sum().item()

        val_acc = val_corrects / len(val_loader.dataset)
        print(f"Val   Acc: {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0
            print("Best DenseNet121 model updated.")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping!")
                break

    return model, best_model_wts

### Train & Save Best Weights

In [ ]:
densenet, best_model_wts = train_model(
    densenet, criterion, optimizer, train_loader, val_loader,
    device, num_epochs=50, patience=5
)

# Save best weights
densenet.load_state_dict(best_model_wts)
torch.save(densenet.state_dict(), "/kaggle/working/densenet121_ham10000.pth")
print("DenseNet121 model saved to /kaggle/working/densenet121_ham10000.pth")

### Reload the Model — DenseNet121

In [ ]:
densenet = densenet121(weights=None)
densenet.classifier = nn.Linear(densenet.classifier.in_features, len(class_names))
densenet.load_state_dict(torch.load("/kaggle/input/densenet121-ham10000-model-weights/densenet121_ham10000.pth", map_location=device))
densenet = densenet.to(device)
densenet.eval()

### Evaluate on Test Set — Accuracy, F1, AUC, Confusion Matrix

In [ ]:
all_preds, all_labels, all_probs = [], [], []

densenet.eval()
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images = images.to(device)
        outputs = densenet(images)
        probs = torch.softmax(outputs, dim=1)
        preds = probs.argmax(1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

acc = accuracy_score(all_labels, all_preds)
f1  = f1_score(all_labels, all_preds, average="weighted")
try:
    auc_score = roc_auc_score(all_labels, all_probs, multi_class="ovr")
except ValueError:
    auc_score = "N/A (AUC needs >1 class in test set)"

print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=class_names))
print(f"Test Accuracy: {acc:.4f}")
print(f"Weighted F1 Score: {f1:.4f}")
print(f"ROC AUC Score: {auc_score}")

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted Label"); plt.ylabel("True Label")
plt.title("Confusion Matrix — DenseNet121")
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix_densenet121.png", dpi=300)
plt.show()
print("Saved: confusion_matrix_densenet121.png")

### ROC‑AUC Curves - DenseNet121

In [ ]:
plot_multiclass_roc(
    all_labels=np.array(all_labels),
    all_probs=np.array(all_probs),
    class_names=class_names,
    title="ROC Curves — DenseNet121",
    outfile="/kaggle/working/roc_curves_densenet121.png"
)

### Grad‑CAM Generation for Test Images — DenseNet‑121 (Plot + Save, with Fallback)

In [ ]:
# Sanity checks
assert 'densenet' in globals(), "densenet model not found. Train/reload DenseNet‑121 first."
assert 'test_loader' in globals(), "test_loader not found."
assert 'class_names' in globals(), "class_names not found."
assert 'device' in globals(), "device not found (cuda/cpu)."

# Inverse of normalize([0.5]*3, [0.5]*3) for correct colors
def inv_norm_05(img_t):
    x = img_t.detach().cpu().clone()
    for c in range(3):
        x[c] = x[c] * 0.5 + 0.5
    return x.clamp(0,1).permute(1,2,0).numpy()  # HWC, [0,1]

# Find the last conv layer (good default CAM target)
def _find_last_conv(m: nn.Module):
    last = None
    for mod in m.modules():
        if isinstance(mod, nn.Conv2d):
            last = mod
    if last is None:
        raise RuntimeError("No Conv2d layer found in DenseNet.")
    return last

densenet.eval()
target_layer = _find_last_conv(densenet)
cam = PTGradCAM(model=densenet, target_layers=[target_layer])

# Collect one image per class (prefer correct predictions), else fallback
got = {c: False for c in class_names}
cam_results = {}

# Pass 1 — correct predictions
for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
    with torch.no_grad():
        preds = densenet(images).argmax(1)

    for i in range(images.size(0)):
        t_lbl = labels[i].item()
        p_lbl = preds[i].item()
        cname = class_names[t_lbl]
        if got[cname] or p_lbl != t_lbl:
            continue

        it = images[i].unsqueeze(0).requires_grad_(True)
        densenet.zero_grad()
        gcam = cam(input_tensor=it, targets=[ClassifierOutputTarget(p_lbl)])[0]

        rgb01 = inv_norm_05(images[i])
        cam_img = show_cam_on_image(rgb01, gcam, use_rgb=True)

        cam_results[cname] = (images[i].cpu(), cam_img)
        got[cname] = True
    if all(got.values()):
        break

# Pass 2 — fallback (first available, even if misclassified)
if not all(got.values()):
    missing = [c for c,v in got.items() if not v]
    print(f"[Info] No correct predictions for classes: {missing}. Falling back to first available sample.")
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        for i in range(images.size(0)):
            t_lbl = labels[i].item()
            cname = class_names[t_lbl]
            if got[cname]:
                continue

            it = images[i].unsqueeze(0).requires_grad_(True)
            densenet.zero_grad()
            gcam = cam(input_tensor=it, targets=[ClassifierOutputTarget(t_lbl)])[0]

            rgb01 = inv_norm_05(images[i])
            cam_img = show_cam_on_image(rgb01, gcam, use_rgb=True)

            cam_results[cname] = (images[i].cpu(), cam_img)
            got[cname] = True
        if all(got.values()):
            break

# Plot & save
out_dir = Path("/kaggle/working/gradcam_densenet121_per_class")
out_dir.mkdir(parents=True, exist_ok=True)

if not cam_results:
    print("[Warning] No Grad‑CAM images generated for DenseNet‑121. Check model/loader.")
else:
    for cname, (img_t, cam_img) in cam_results.items():
        orig = inv_norm_05(img_t)
        plt.figure(figsize=(8,4))
        plt.subplot(1,2,1); plt.imshow(orig);   plt.title(f"Original — {cname}"); plt.axis('off')
        plt.subplot(1,2,2); plt.imshow(cam_img); plt.title("Grad‑CAM");           plt.axis('off')
        plt.tight_layout()
        save_path = out_dir / f"gradcam_{cname}.png"
        plt.savefig(save_path, dpi=300, bbox_inches="tight", pad_inches=0.05)
        plt.show()
    print(f"Saved DenseNet‑121 Grad‑CAM images to: {out_dir.resolve()}")

# Cleanup hooks
try:
    cam.activations_and_grads.release()
except Exception:
    pass
del cam

## EfficientNet-B3

### Add/Ensure Image Paths — EfficientNet‑B3

In [ ]:
# Define image directories for HAM10000
image_dir_1 = "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1/"
image_dir_2 = "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2/"

# Function to resolve full image path
def resolve_path(img_id):
    f1 = os.path.join(image_dir_1, f"{img_id}.jpg")
    f2 = os.path.join(image_dir_2, f"{img_id}.jpg")
    return f1 if os.path.exists(f1) else f2

# Apply to existing 'image_id' column
df["path"] = df["image_id"].apply(resolve_path)

# Check results
print("Sample resolved paths:")
print(df[["image_id", "path"]].head())

### Transforms — EfficientNet‑B3 (300×300)

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

val_transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

### Define SkinLesionDataset Class

In [ ]:
class SkinLesionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = row["label"]
        return image, label

### Validate and Clean Image Paths

In [ ]:
# Check for missing image paths
df["path"] = df["image_id"].apply(resolve_path)  # <- use image_id here
df["file_exists"] = df["path"].apply(lambda p: os.path.exists(p))

# Drop rows where the image file does not exist
df = df[df["file_exists"]].reset_index(drop=True)

# Drop the helper column
df = df.drop(columns=["file_exists"])

### Train/Val/Test Split & DataLoaders — EfficientNet‑B3

In [ ]:
# 70/15/15 or 80/10/10
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42)

train_dataset = SkinLesionDataset(train_df, transform=train_transform)
val_dataset   = SkinLesionDataset(val_df,   transform=val_transform)
test_dataset  = SkinLesionDataset(test_df,  transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

### EfficientNet‑B3 Initialization (TorchVision)

In [ ]:
# Set number of output classes
num_classes = 7  # or use len(class_names) if already defined

# Initialize EfficientNet-B3 without pretrained weights (no download)
effnet = efficientnet_b3(weights=None)

# Replace classifier layer
effnet.classifier[1] = nn.Linear(effnet.classifier[1].in_features, num_classes)

# Move to GPU if available
effnet = effnet.to(device)

# Define loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(effnet.parameters(), lr=1e-4)

### Train Function — EfficientNet‑B3 (Early Stopping)

In [ ]:
def train_model_effnet(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=50, patience=5):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    patience_counter = 0

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        model.train()
        running_loss, running_corrects = 0.0, 0

        for inputs, labels in tqdm(train_loader, desc="Training", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            preds = outputs.argmax(1)
            running_loss += loss.item() * inputs.size(0)
            running_corrects += (preds == labels).sum().item()

        train_loss = running_loss / len(train_loader.dataset)
        train_acc  = running_corrects / len(train_loader.dataset)

        # Validation phase
        model.eval()
        val_corrects = 0
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc="Validating", leave=False):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                preds = outputs.argmax(1)
                val_corrects += (preds == labels).sum().item()

        val_acc = val_corrects / len(val_loader.dataset)
        print(f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping!")
                break

    return model, best_model_wts

###  Train & Save the Best EfficientNet‑B3

In [ ]:
# Train EfficientNet-B3 on HAM10000
effnet, best_wts = train_model_effnet(
    effnet, train_loader, val_loader, criterion, optimizer, device,
    num_epochs=50, patience=5
)

# Load best weights
effnet.load_state_dict(best_wts)

# Save to disk
torch.save(effnet.state_dict(), "/kaggle/working/efficientnet_b3_best.pth")
print("EfficientNet‑B3 model saved to /kaggle/working/efficientnet_b3_best.pth")

### Reload the Model — EfficientNet‑B3

In [ ]:
# Define class names again
class_names = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']

# Reload the Model — EfficientNet‑B3
effnet = efficientnet_b3(weights=None)
effnet.classifier[1] = nn.Linear(effnet.classifier[1].in_features, len(class_names))
effnet.load_state_dict(torch.load("/kaggle/input/efficientnetb3-ham10000-model-weights/efficientnet_b3_best.pth", map_location=device))
effnet = effnet.to(device)
effnet.eval()

### Evaluate on Test Set — EfficientNet‑B3

In [ ]:
all_preds, all_labels, all_probs = [], [], []

effnet.eval()
with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="Testing"):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = effnet(inputs)
        probs = torch.softmax(outputs, dim=1)
        preds = probs.argmax(1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

acc = accuracy_score(all_labels, all_preds)
f1  = f1_score(all_labels, all_preds, average="weighted")
try:
    auc_score = roc_auc_score(all_labels, all_probs, multi_class="ovr")
except ValueError:
    auc_score = "N/A (AUC needs >1 class in test set)"

print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=class_names))
print(f"Test Accuracy: {acc:.4f}")
print(f"Weighted F1 Score: {f1:.4f}")
print(f"ROC AUC Score: {auc_score}")

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Confusion Matrix — EfficientNet‑B3")
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix_efficientnetb3.png", dpi=300)
plt.show()

### ROC‑AUC Curves — EfficientNet‑B3

In [ ]:
plot_multiclass_roc(
    all_labels=all_labels,
    all_probs=all_probs,
    class_names=class_names,
    title="ROC Curves — EfficientNet‑B3",
    outfile="/kaggle/working/roc_curves_efficientnetb3.png"
)

### Grad‑CAM Generation for Test Images — EfficientNet‑B3 (Plot + Save, with Fallback)

In [ ]:
# Pick objects / sanity checks
effnet_model = globals().get("effnet", None) or globals().get("effnet_model", None)
assert effnet_model is not None, "EfficientNet-B3 model not found."
test_loader_effnet = globals().get("test_loader_effnet", None) or globals().get("test_loader", None)
assert test_loader_effnet is not None, "Test loader not found."
assert 'class_names' in globals(), "class_names not found."
assert 'device' in globals(), "device not found."

# Helper functions
def inv_norm_05(img_t):
    """Invert Normalize([0.5]*3,[0.5]*3) to RGB [0,1]. img_t: CHW tensor."""
    x = img_t.detach().cpu().clone()
    for c in range(3):
        x[c] = x[c] * 0.5 + 0.5
    return x.clamp(0, 1).permute(1, 2, 0).numpy()

def find_last_conv_module(net: nn.Module):
    last = None
    for m in net.modules():
        if isinstance(m, nn.Conv2d):
            last = m
    if last is None:
        raise RuntimeError("No Conv2d layer found in the model.")
    return last

# Setup Grad-CAM
effnet_model.eval()
target_layer = find_last_conv_module(effnet_model)
cam = PTGradCAM(model=effnet_model, target_layers=[target_layer])

got = {c: False for c in class_names}
cam_results = {}

# Pass 1: prefer correctly predicted examples
for images, labels in test_loader_effnet:
    images, labels = images.to(device), labels.to(device)
    with torch.no_grad():
        preds = effnet_model(images).argmax(1)

    for i in range(images.size(0)):
        t_lbl = labels[i].item()
        p_lbl = preds[i].item()
        cname = class_names[t_lbl]
        if got[cname] or p_lbl != t_lbl:
            continue

        it = images[i].unsqueeze(0).requires_grad_(True)
        effnet_model.zero_grad()
        grayscale_cam = cam(input_tensor=it, targets=[ClassifierOutputTarget(p_lbl)])[0]

        rgb01 = inv_norm_05(images[i])
        cam_img = show_cam_on_image(rgb01, grayscale_cam, use_rgb=True)

        cam_results[cname] = (rgb01, cam_img)
        got[cname] = True

    if all(got.values()):
        break

# Pass 2: fallback for missing classes
if not all(got.values()):
    missing = [c for c, v in got.items() if not v]
    print(f"[Info] No correct predictions for classes: {missing}. Falling back to first available sample.")
    for images, labels in test_loader_effnet:
        images, labels = images.to(device), labels.to(device)
        for i in range(images.size(0)):
            t_lbl = labels[i].item()
            cname = class_names[t_lbl]
            if got[cname]:
                continue

            it = images[i].unsqueeze(0).requires_grad_(True)
            effnet_model.zero_grad()
            grayscale_cam = cam(input_tensor=it, targets=[ClassifierOutputTarget(t_lbl)])[0]

            rgb01 = inv_norm_05(images[i])
            cam_img = show_cam_on_image(rgb01, grayscale_cam, use_rgb=True)

            cam_results[cname] = (rgb01, cam_img)
            got[cname] = True

        if all(got.values()):
            break

# Plot & save
out_dir = Path("/kaggle/working/gradcam_efficientnet_b3_per_class")
out_dir.mkdir(parents=True, exist_ok=True)

if not cam_results:
    print("[Warning] No Grad-CAM images generated for EfficientNet-B3.")
else:
    for cname, (orig_rgb01, cam_img) in cam_results.items():
        plt.figure(figsize=(8, 4))
        plt.subplot(1, 2, 1); plt.imshow(orig_rgb01); plt.title(f"Original — {cname}"); plt.axis('off')
        plt.subplot(1, 2, 2); plt.imshow(cam_img);     plt.title("Grad-CAM");          plt.axis('off')
        plt.tight_layout()
        sp = out_dir / f"gradcam_{cname}.png"
        plt.savefig(sp, dpi=300, bbox_inches="tight", pad_inches=0.05)
        plt.show()
    print(f"Saved EfficientNet-B3 Grad-CAM images to: {out_dir.resolve()}")

# Cleanup hooks
try:
    cam.activations_and_grads.release()
except Exception:
    pass
del cam

## ConvNeXt-Tiny

### Reproducibility & ImageNet Normalization Setup

In [ ]:
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ImageNet stats (ConvNeXt expects these)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

### Dataset, Fixed Label Map, Splits, Transforms, Loaders

In [ ]:
# Dataset class that reads PIL images from df['path'] and returns (tensor, label)
class SkinLesionDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        p = self.df.loc[idx, "path"]
        y = int(self.df.loc[idx, "label"])
        img = Image.open(p).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, y

# Resolve image paths from both folders
img_dir1 = "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1"
img_dir2 = "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2"

def resolve_path(img_id):
    p1 = os.path.join(img_dir1, f"{img_id}.jpg")
    p2 = os.path.join(img_dir2, f"{img_id}.jpg")
    return p1 if os.path.exists(p1) else p2

# Load metadata
df = pd.read_csv("/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_metadata.csv")
df["path"] = df["image_id"].apply(resolve_path)

# Use a stable label mapping (alphabetical is fine)
class_names = sorted(df["dx"].unique().tolist())
label_map = {name: i for i, name in enumerate(class_names)}
df["label"] = df["dx"].map(label_map).astype(int)

# Split 70/15/15 stratified
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42
)

print("Split sizes:", len(train_df), len(val_df), len(test_df))
print("Classes:", class_names)

# Use ImageNet normalization everywhere for ConvNeXt
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Datasets & loaders
train_ds = SkinLesionDataset(train_df, transform=train_tf)
val_ds   = SkinLesionDataset(val_df,   transform=eval_tf)
test_ds  = SkinLesionDataset(test_df,  transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

### Model Init (ConvNeXt‑Tiny), Loss, Optimizer, Scheduler

In [ ]:
convnext = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
convnext.classifier[2] = nn.Linear(convnext.classifier[2].in_features, len(class_names))
convnext = convnext.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(convnext.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=2, factor=0.5, verbose=True
)

### Train + Early Stopping (AMP), Save Best

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer,
                device, num_epochs=50, patience=5, scheduler=None, use_amp=True):
    best_wts = deepcopy(model.state_dict())
    best_acc = 0.0
    no_improve = 0

    scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device.type == "cuda"))

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")

        # Train
        model.train()
        tr_loss, tr_correct, tr_seen = 0.0, 0, 0
        for x, y in tqdm(train_loader, desc="Training", leave=False):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=scaler.is_enabled()):
                logits = model(x)
                loss = criterion(logits, y)

            if scaler.is_enabled():
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()

            preds = logits.argmax(1)
            tr_loss   += loss.item() * x.size(0)
            tr_correct+= (preds == y).sum().item()
            tr_seen   += y.size(0)

        train_loss = tr_loss / tr_seen
        train_acc  = tr_correct / tr_seen

        # Validate
        model.eval()
        val_correct, val_seen = 0, 0
        with torch.no_grad():
            for x, y in tqdm(val_loader, desc="Validating", leave=False):
                x, y = x.to(device), y.to(device)
                logits = model(x)
                val_correct += (logits.argmax(1) == y).sum().item()
                val_seen    += y.size(0)

        val_acc = val_correct / val_seen
        if scheduler is not None:
            scheduler.step(val_acc)

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val   Acc: {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            best_wts = deepcopy(model.state_dict())
            no_improve = 0
            print("New best model (by Val Acc).")
        else:
            no_improve += 1
            if no_improve >= patience:
                print("Early stopping triggered.")
                break

    model.load_state_dict(best_wts)
    print(f"Best Val Acc: {best_acc:.4f}")

    # Save best
    save_path = "/kaggle/working/convnext_tiny_ham10000.pth"
    torch.save(model.state_dict(), save_path)
    print(f"Saved best model to {save_path}")
    return model

convnext = train_model(
    convnext, train_loader, val_loader, criterion, optimizer,
    device, num_epochs=50, patience=5, scheduler=scheduler, use_amp=True
)

### Reload Best & Evaluate (VAL Sanity then TEST)

In [ ]:
# Reload to be extra sure we use the saved best
convnext_loaded = convnext_tiny(weights=None)
convnext_loaded.classifier[2] = nn.Linear(convnext_loaded.classifier[2].in_features, len(class_names))
convnext_loaded.load_state_dict(torch.load("/kaggle/input/ham10000-convnext-tiny-weights/convnext_tiny_ham10000.pth", map_location=device))
convnext_loaded = convnext_loaded.to(device)
convnext_loaded.eval()

def evaluate(model, loader, device, name="SPLIT"):
    all_preds, all_labels, all_probs = [], [], []
    model.eval()
    with torch.no_grad():
        for x, y in tqdm(loader, desc=f"Testing ({name})"):
            x, y = x.to(device), y.to(device)
            logits = model(x)
            probs  = torch.softmax(logits, dim=1)
            preds  = probs.argmax(1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)

    print(f"\n{name} — Classification Report:\n")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average="weighted")
    try:
        auc_score = roc_auc_score(all_labels, all_probs, multi_class="ovr")
    except ValueError:
        auc_score = "N/A (needs >1 class present)"
    print(f"{name} — Acc: {acc:.4f} | F1 (weighted): {f1:.4f} | AUC (OvR): {auc_score}")

    # Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.title(f"Confusion Matrix — ConvNeXt‑Tiny ({name})")
    plt.tight_layout()
    out_png = f"/kaggle/working/confusion_matrix_convnext_tiny_{name.lower()}.png"
    plt.savefig(out_png, dpi=300)
    plt.show()
    print(f"Saved: {out_png}")

    return acc, f1, auc_score, all_labels, all_probs

# Final numbers on test
acc, f1, auc_score, all_labels, all_probs = evaluate(convnext_loaded, test_loader, device, name="TEST")

### ROC‑AUC Curves — ConvNeXt‑Tiny

In [ ]:
plot_multiclass_roc(
    all_labels=all_labels,
    all_probs=all_probs,
    class_names=class_names,
    title="ROC Curves — ConvNeXt‑Tiny",
    outfile="/kaggle/working/roc_curves_convnext_tiny.png"
)

### Grad‑CAM Generation for Test Images — ConvNeXt‑Tiny (Plot + Save, with Fallback)

In [ ]:
# Pick objects / sanity checks
convnext_model = globals().get("convnext_loaded", None) or globals().get("convnext", None)
assert convnext_model is not None, "ConvNeXt‑Tiny model not found (convnext_loaded or convnext)."
test_loader_convnext = globals().get("test_loader_convnext", None) or globals().get("test_loader", None)
assert test_loader_convnext is not None, "Test loader not found (test_loader_convnext or test_loader)."
assert 'class_names' in globals(), "class_names not found."
assert 'device' in globals(), "device not found."

# Helpers
def inv_norm_05(img_t):
    """Invert Normalize([0.5]*3,[0.5]*3) -> RGB [0,1]. img_t: CHW tensor."""
    x = img_t.detach().cpu().clone()
    for c in range(3):
        x[c] = x[c] * 0.5 + 0.5
    return x.clamp(0, 1).permute(1, 2, 0).numpy()

def find_last_conv_module(net: nn.Module):
    last = None
    for m in net.modules():
        if isinstance(m, nn.Conv2d):
            last = m
    if last is None:
        raise RuntimeError("No Conv2d layer found in ConvNeXt‑Tiny.")
    return last

# Setup Grad‑CAM
convnext_model.eval()
target_layer = find_last_conv_module(convnext_model)  # safe default
cam = PTGradCAM(model=convnext_model, target_layers=[target_layer])

got = {c: False for c in class_names}
cam_results = {}

# Pass 1: prefer correctly predicted examples
for images, labels in test_loader_convnext:
    images, labels = images.to(device), labels.to(device)
    with torch.no_grad():
        preds = convnext_model(images).argmax(1)

    for i in range(images.size(0)):
        t_lbl = labels[i].item()
        p_lbl = preds[i].item()
        cname = class_names[t_lbl]
        if got[cname] or p_lbl != t_lbl:
            continue

        it = images[i].unsqueeze(0).requires_grad_(True)
        convnext_model.zero_grad()
        grayscale_cam = cam(input_tensor=it, targets=[ClassifierOutputTarget(p_lbl)])[0]

        rgb01 = inv_norm_05(images[i])
        cam_img = show_cam_on_image(rgb01, grayscale_cam, use_rgb=True)

        cam_results[cname] = (rgb01, cam_img)
        got[cname] = True

    if all(got.values()):
        break

# Pass 2: fallback for missing classes
if not all(got.values()):
    missing = [c for c, v in got.items() if not v]
    print(f"[Info] No correct predictions for classes: {missing}. Falling back to first available sample.")
    for images, labels in test_loader_convnext:
        images, labels = images.to(device), labels.to(device)
        for i in range(images.size(0)):
            t_lbl = labels[i].item()
            cname = class_names[t_lbl]
            if got[cname]:
                continue

            it = images[i].unsqueeze(0).requires_grad_(True)
            convnext_model.zero_grad()
            # drive CAM toward the TRUE label even if misclassified
            grayscale_cam = cam(input_tensor=it, targets=[ClassifierOutputTarget(t_lbl)])[0]

            rgb01 = inv_norm_05(images[i])
            cam_img = show_cam_on_image(rgb01, grayscale_cam, use_rgb=True)

            cam_results[cname] = (rgb01, cam_img)
            got[cname] = True

        if all(got.values()):
            break

# Plot & save
out_dir = Path("/kaggle/working/gradcam_convnext_tiny_per_class")
out_dir.mkdir(parents=True, exist_ok=True)

if not cam_results:
    print("[Warning] No Grad‑CAM images generated for ConvNeXt‑Tiny.")
else:
    for cname, (orig_rgb01, cam_img) in cam_results.items():
        plt.figure(figsize=(8, 4))
        plt.subplot(1, 2, 1); plt.imshow(orig_rgb01); plt.title(f"Original — {cname}"); plt.axis('off')
        plt.subplot(1, 2, 2); plt.imshow(cam_img);     plt.title("Grad‑CAM");          plt.axis('off')
        plt.tight_layout()
        sp = out_dir / f"gradcam_{cname}.png"
        plt.savefig(sp, dpi=300, bbox_inches="tight", pad_inches=0.05)
        plt.show()
    print(f"Saved ConvNeXt‑Tiny Grad‑CAM images to: {out_dir.resolve()}")

# Cleanup hooks
try:
    cam.activations_and_grads.release()
except Exception:
    pass
del cam

## MobileNetV3

### MobileNetV3 — Load Metadata & Resolve Image Paths

In [ ]:
df = pd.read_csv("/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_metadata.csv")

img_dir_1 = "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1"
img_dir_2 = "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2"

def resolve_path(img_id):
    p1 = os.path.join(img_dir_1, f"{img_id}.jpg")
    p2 = os.path.join(img_dir_2, f"{img_id}.jpg")
    return p1 if os.path.exists(p1) else p2

df["path"] = df["image_id"].apply(resolve_path)
df["label"] = df["dx"].map(label_map).astype(int)

### MobileNetV3 — Train/Val/Test Split

In [ ]:
df_temp, test_df = train_test_split(df, test_size=0.20, stratify=df["label"], random_state=42)
train_df, val_df = train_test_split(df_temp, test_size=0.20, stratify=df_temp["label"], random_state=42)

print(len(train_df), len(val_df), len(test_df))

### MobileNetV3 — Transforms

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

### MobileNetV3 — Dataloaders

In [ ]:
train_dataset = SkinLesionDataset(train_df, transform=train_transform)
val_dataset   = SkinLesionDataset(val_df,   transform=val_transform)
test_dataset  = SkinLesionDataset(test_df,  transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

### Model Definition

In [ ]:
# Initialize MobileNetV3 large with pretrained weights
mobilenet = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1)

# Modify final classifier to match HAM10000 number of classes (7)
mobilenet.classifier[3] = nn.Linear(mobilenet.classifier[3].in_features, 7)

mobilenet = mobilenet.to(device)

# Define loss & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mobilenet.parameters(), lr=1e-4)

### MobileNetV3 — Train (Early Stopping, Save Best)

In [ ]:
def train_mobilenet(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=50, patience=5):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        model.train()
        run_loss, run_correct, run_total = 0.0, 0, 0

        for inputs, labels in tqdm(train_loader, desc="Training", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            run_loss += loss.item() * inputs.size(0)
            run_correct += (outputs.argmax(1) == labels).sum().item()
            run_total += labels.size(0)

        train_loss = run_loss / run_total
        train_acc  = run_correct / run_total

        # Validation
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc="Validating", leave=False):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_total   += labels.size(0)

        val_acc = val_correct / val_total
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val   Acc: {val_acc:.4f}")

        # Early stopping
        if val_acc > best_acc:
            best_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(best_model_wts, "/kaggle/working/mobilenetv3_ham10000.pth")
            print("Best MobileNetV3 model saved.")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered.")
                break

    return model, best_model_wts

mobilenet, best_model_wts = train_mobilenet(
    mobilenet, train_loader, val_loader, criterion, optimizer, device,
    num_epochs=50, patience=5
)
# Save final best
mobilenet.load_state_dict(best_model_wts)
torch.save(mobilenet.state_dict(), "/kaggle/working/mobilenetv3_ham10000.pth")
print("MobileNetV3 model saved to /kaggle/working/mobilenetv3_ham10000.pth")

### Reload the Model — MobileNetV3

In [ ]:
mobilenet = mobilenet_v3_large(weights=None)
mobilenet.classifier[3] = nn.Linear(mobilenet.classifier[3].in_features, len(class_names))
mobilenet.load_state_dict(torch.load("//kaggle/input/mobilenetv3-ham10000-model-weights/mobilenetv3_ham10000.pth", map_location=device))
mobilenet = mobilenet.to(device)
mobilenet.eval()

### Evaluate on Test Set — Metrics & Confusion Matrix (MobileNetV3)

In [ ]:
all_preds_mnv3, all_labels_mnv3, all_probs_mnv3 = [], [], []

mobilenet.eval()
with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="Testing"):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = mobilenet(inputs)
        probs = torch.softmax(outputs, dim=1)

        all_preds_mnv3.extend(probs.argmax(1).cpu().numpy())
        all_labels_mnv3.extend(labels.cpu().numpy())
        all_probs_mnv3.extend(probs.cpu().numpy())

# Convert to arrays
all_preds_mnv3  = np.array(all_preds_mnv3)
all_labels_mnv3 = np.array(all_labels_mnv3)
all_probs_mnv3  = np.array(all_probs_mnv3)

# Metrics
acc = accuracy_score(all_labels_mnv3, all_preds_mnv3)
f1  = f1_score(all_labels_mnv3, all_preds_mnv3, average="weighted")
try:
    auc_score = roc_auc_score(all_labels_mnv3, all_probs_mnv3, multi_class="ovr")
except ValueError:
    auc_score = "N/A (AUC needs >1 class in test set)"

# Report
print("\nClassification Report:\n")
print(classification_report(all_labels_mnv3, all_preds_mnv3, target_names=class_names))
print(f"Test Accuracy: {acc:.4f}")
print(f"Weighted F1 Score: {f1:.4f}")
print(f"ROC AUC Score: {auc_score}")

# Confusion Matrix
cm = confusion_matrix(all_labels_mnv3, all_preds_mnv3)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix — MobileNetV3")
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix_mobilenetv3.png", dpi=300)
plt.show()
print("Confusion matrix saved to 'confusion_matrix_mobilenetv3.png'.")

### ROC‑AUC Curves — MobileNetV3

In [ ]:
plot_multiclass_roc(
    all_labels=np.array(all_labels_mnv3),
    all_probs=np.array(all_probs_mnv3),
    class_names=class_names,
    title="ROC Curves — MobileNetV3",
    outfile="/kaggle/working/roc_curves_mobilenetv3.png"
)

### Grad‑CAM Generation for Test Images — MobileNetV3‑Large (Plot + Save, with Fallback)

In [ ]:
# Pick objects / sanity checks
mobilenet_model = globals().get("mobilenet", None)
assert mobilenet_model is not None, "MobileNetV3 model not found (variable 'mobilenet')."
test_loader_mnv3 = globals().get("test_loader_mnv3", None) or globals().get("test_loader", None)
assert test_loader_mnv3 is not None, "Test loader not found (test_loader_mnv3 or test_loader)."
assert 'class_names' in globals(), "class_names not found."
assert 'device' in globals(), "device not found."

# Helpers
def inv_norm_05(img_t):
    """Invert Normalize([0.5]*3,[0.5]*3) -> RGB [0,1]. img_t: CHW tensor."""
    x = img_t.detach().cpu().clone()
    for c in range(3):
        x[c] = x[c] * 0.5 + 0.5
    return x.clamp(0, 1).permute(1, 2, 0).numpy()

def find_last_conv_module(net: nn.Module):
    last = None
    for m in net.modules():
        if isinstance(m, nn.Conv2d):
            last = m
    if last is None:
        raise RuntimeError("No Conv2d layer found in MobileNetV3‑Large.")
    return last

# Setup Grad‑CAM
mobilenet_model.eval()
target_layer = find_last_conv_module(mobilenet_model)  # last conv in features
cam = PTGradCAM(model=mobilenet_model, target_layers=[target_layer])

got = {c: False for c in class_names}
cam_results = {}

# Pass 1: prefer correctly predicted examples
for images, labels in test_loader_mnv3:
    images, labels = images.to(device), labels.to(device)
    with torch.no_grad():
        preds = mobilenet_model(images).argmax(1)

    for i in range(images.size(0)):
        t_lbl = labels[i].item()
        p_lbl = preds[i].item()
        cname = class_names[t_lbl]
        if got[cname] or p_lbl != t_lbl:
            continue

        it = images[i].unsqueeze(0).requires_grad_(True)
        mobilenet_model.zero_grad()
        grayscale_cam = cam(input_tensor=it, targets=[ClassifierOutputTarget(p_lbl)])[0]

        rgb01 = inv_norm_05(images[i])
        cam_img = show_cam_on_image(rgb01, grayscale_cam, use_rgb=True)

        cam_results[cname] = (rgb01, cam_img)
        got[cname] = True

    if all(got.values()):
        break

# Pass 2: fallback for missing classes
if not all(got.values()):
    missing = [c for c, v in got.items() if not v]
    print(f"[Info] No correct predictions for classes: {missing}. Falling back to first available sample.")
    for images, labels in test_loader_mnv3:
        images, labels = images.to(device), labels.to(device)
        for i in range(images.size(0)):
            t_lbl = labels[i].item()
            cname = class_names[t_lbl]
            if got[cname]:
                continue

            it = images[i].unsqueeze(0).requires_grad_(True)
            mobilenet_model.zero_grad()
            # drive CAM toward the TRUE label even if misclassified
            grayscale_cam = cam(input_tensor=it, targets=[ClassifierOutputTarget(t_lbl)])[0]

            rgb01 = inv_norm_05(images[i])
            cam_img = show_cam_on_image(rgb01, grayscale_cam, use_rgb=True)

            cam_results[cname] = (rgb01, cam_img)
            got[cname] = True

        if all(got.values()):
            break

# Plot & save
out_dir = Path("/kaggle/working/gradcam_mobilenet_v3_per_class")
out_dir.mkdir(parents=True, exist_ok=True)

if not cam_results:
    print("[Warning] No Grad‑CAM images generated for MobileNetV3‑Large.")
else:
    for cname, (orig_rgb01, cam_img) in cam_results.items():
        plt.figure(figsize=(8, 4))
        plt.subplot(1, 2, 1); plt.imshow(orig_rgb01); plt.title(f"Original — {cname}"); plt.axis('off')
        plt.subplot(1, 2, 2); plt.imshow(cam_img);     plt.title("Grad‑CAM");          plt.axis('off')
        plt.tight_layout()
        sp = out_dir / f"gradcam_{cname}.png"
        plt.savefig(sp, dpi=300, bbox_inches="tight", pad_inches=0.05)
        plt.show()
    print(f"Saved MobileNetV3‑Large Grad‑CAM images to: {out_dir.resolve()}")

# Cleanup hooks
try:
    cam.activations_and_grads.release()
except Exception:
    pass
del cam

## 2.4 Ensemble Framework

### Define CNNModel Architecture

In [ ]:
class CNNModel(nn.Module):
    def __init__(self, num_classes=7):
        super(CNNModel, self).__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        # 224x224 -> after 4 pools -> 14x14 feature map with 256 channels
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

### Ensemble — Reload All Models

In [ ]:
# Helper: resolve image path from image_id (fallback)
IMG_DIRS = [
    "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1",
    "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2",
]

def find_image_path_from_id(image_id: str):
    fname = f"{image_id}.jpg"
    for d in IMG_DIRS:
        p = os.path.join(d, fname)
        if os.path.exists(p):
            return p
    return None

num_classes = len(class_names)

# CNN
cnn_model = CNNModel(num_classes=num_classes)
cnn_model.load_state_dict(torch.load(
    "/kaggle/input/kaggleinputham10000-model-weights/cnn_ham10000_best_model.pth",
    map_location=device
))
cnn_model = cnn_model.to(device).eval()

# ResNet-50

resnet_model = resnet50(weights=None)  # avoid default weights
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, num_classes)

resnet_model.load_state_dict(torch.load(
    "/kaggle/input/resnet50-ham10000-model-weights/resnet50_ham10000.pth", 
    map_location=device
))

resnet_model = resnet_model.to(device).eval()

# DenseNet121
densenet_model = densenet121(weights=None)
densenet_model.classifier = nn.Linear(densenet_model.classifier.in_features, num_classes)
densenet_model.load_state_dict(torch.load(
    "/kaggle/input/densenet121-ham10000-model-weights/densenet121_ham10000.pth", 
    map_location=device))
densenet_model = densenet_model.to(device).eval()

# EfficientNet-B3
effnet_model = efficientnet_b3(weights=None)
effnet_model.classifier[1] = nn.Linear(effnet_model.classifier[1].in_features, num_classes)
effnet_model.load_state_dict(torch.load("/kaggle/input/efficientnetb3-ham10000-model-weights/efficientnet_b3_best.pth", map_location=device))
effnet_model = effnet_model.to(device).eval()

# ConvNeXt-Tiny
convnext_model = convnext_tiny(weights=None)
convnext_model.classifier[2] = nn.Linear(convnext_model.classifier[2].in_features, num_classes)
convnext_model.load_state_dict(torch.load("/kaggle/input/ham10000-convnext-tiny-weights/convnext_tiny_ham10000.pth", map_location=device))
convnext_model = convnext_model.to(device).eval()

# MobileNetV3-Large
mobilenet_model = mobilenet_v3_large(weights=None)
mobilenet_model.classifier[3] = nn.Linear(mobilenet_model.classifier[3].in_features, num_classes)
mobilenet_model.load_state_dict(torch.load("/kaggle/input/mobilenetv3-ham10000-model-weights/mobilenetv3_ham10000.pth", map_location=device))
mobilenet_model = mobilenet_model.to(device).eval()

# ViT (vit-base-patch16-224)
# Define correct config
vit_config = ViTConfig.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=num_classes  # must be 7
)

# Instantiate the model with random weights (but correct output shape)
vit_model = ViTForImageClassification(config=vit_config)

# Load fine-tuned weights
vit_model.load_state_dict(torch.load(
    "/kaggle/input/vit-ham10000-model-weights/vit_ham10000_best_model.pth",
    map_location=device
))

vit_model = vit_model.to(device).eval()

### Ensemble — Preprocessing Utilities (Per‑Model)

In [ ]:
# Common transforms
to_tensor = transforms.ToTensor()
resize_224 = transforms.Resize((224, 224))
resize_300 = transforms.Resize((300, 300))

normalize_05 = transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
normalize_imagenet = transforms.Normalize([0.485, 0.456, 0.406],
                                          [0.229, 0.224, 0.225])

# CNN, DenseNet, ConvNeXt, MobileNetV3 used 224 with 0.5/0.5 norm in the flows
preprocess_224_05 = transforms.Compose([resize_224, to_tensor, normalize_05])

# ResNet-50 used ImageNet stats (224)
preprocess_resnet = transforms.Compose([resize_224, to_tensor, normalize_imagenet])

# EfficientNet-B3 used 300 with 0.5/0.5 norm
preprocess_effnet = transforms.Compose([resize_300, to_tensor, normalize_05])

# ViT uses its processor; recreate it
vit_processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")

def preprocess_vit(pil_img: Image.Image) -> torch.Tensor:
    enc = vit_processor(images=pil_img, return_tensors="pt")
    return enc["pixel_values"].squeeze(0)  # CHW

### Reconstruct Stratified Test Set (2003 Images)

In [ ]:
# Load metadata
meta_csv = "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_metadata.csv"
df = pd.read_csv(meta_csv)

# Ensure label column matches with model labels
df = df.rename(columns={"dx": "label"})

# Encode labels
label_map = {l: i for i, l in enumerate(sorted(df["label"].unique()))}
df["label"] = df["label"].map(label_map)

# Resolve image paths
IMG_DIRS = [
    "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1",
    "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2",
]

def find_image_path(image_id):
    for d in IMG_DIRS:
        p = os.path.join(d, f"{image_id}.jpg")
        if os.path.exists(p):
            return p
    return None

df["path"] = df["image_id"].apply(find_image_path)
df = df[df["path"].notna()].reset_index(drop=True)

# Stratified split (10% test = ~2003 samples)
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
_, test_idx = next(sss.split(df["image_id"], df["label"]))
test_df = df.iloc[test_idx].reset_index(drop=True)

# Save for reuse (optional)
test_df.to_csv("/kaggle/working/test_balanced.csv", index=False)
print("Reconstructed test set:", test_df.shape)

# 	3.	Evaluation Metrics

## 3.1 Classification Metrics

### Compare All Individual Models and Ensemble

In [ ]:
assert 'class_names' in globals() and 'device' in globals(), "Define class_names and device first."
NUM_C = len(class_names)

# Ensure test_df exists (load if missing)
if 'test_df' not in globals():
    # Candidate CSVs (adjust if the path differs)
    CANDIDATE_TEST_CSVS = [
        "/kaggle/input/ham10000-balanced-split/test_balanced.csv",
        "/kaggle/input/ham10000-test-balanced-csv/test_balanced.csv",
        "/kaggle/working/test_balanced.csv",
    ]
    test_csv_path = next((p for p in CANDIDATE_TEST_CSVS if os.path.exists(p)), None)
    assert test_csv_path, f"Could not find test_balanced.csv in: {CANDIDATE_TEST_CSVS}"
    test_df = pd.read_csv(test_csv_path)

    # Ensure a usable 'path' column
    if "path" not in test_df.columns or test_df["path"].isna().any():
        assert "image_id" in test_df.columns, "test_balanced.csv must have 'path' or 'image_id'."
        assert 'find_image_path_from_id' in globals(), "Define find_image_path_from_id first."
        test_df["path"] = test_df["image_id"].apply(find_image_path_from_id)

    # Keep only rows that actually exist on disk
    test_df = test_df[test_df["path"].apply(lambda p: isinstance(p, str) and os.path.exists(p))].reset_index(drop=True)

print(f"Test rows with valid images: {len(test_df)}")

# Build a subset to infer class-index permutations for ResNet-50 & ViT 
def _valid_rows(df):
    rows = []
    for _, r in df.iterrows():
        y = int(r["label"])
        p = r["path"]
        if isinstance(p, str) and os.path.exists(p):
            rows.append((y, p))
    return rows

pool = _valid_rows(test_df)
if len(pool) < 100:
    # shuffle to increase chance of diverse classes
    pool = _valid_rows(test_df.sample(frac=1.0, random_state=42).reset_index(drop=True))
sample_for_perm = pool[:400]

def infer_perm_on_subset(model_name):
    """
    Returns (mapping_dict, reorder_fn).
    mapping_dict: {our_idx -> model_idx}
    reorder_fn(p_model): returns probs re-ordered to match our class_names.
    """
    counts = np.zeros((NUM_C, NUM_C), dtype=np.int64)  # rows=true(our), cols=pred(model)

    for (y_true, path) in tqdm(sample_for_perm, desc=f"[perm] {model_name}", leave=False):
        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            continue

        with torch.no_grad():
            if model_name == "resnet50":
                x = preprocess_resnet(img).unsqueeze(0).to(device)
                pred = int(torch.softmax(resnet_model(x), dim=1).argmax(1).item())
            elif model_name == "vit":
                x = preprocess_vit(img).unsqueeze(0).to(device)
                pred = int(torch.softmax(vit_model(x).logits, dim=1).argmax(1).item())
            else:
                raise ValueError("Only resnet50 or vit supported here.")

        if 0 <= y_true < NUM_C and 0 <= pred < NUM_C:
            counts[y_true, pred] += 1

    try:
        # maximize diagonal
        r_idx, c_idx = linear_sum_assignment(-counts)
    except Exception:
        # Greedy fallback
        used_true = set(); r_idx = []; c_idx = []
        for c in range(NUM_C):
            r = int(np.argmax(counts[:, c]))
            if r not in used_true:
                r_idx.append(r); c_idx.append(c); used_true.add(r)
            if len(used_true) == NUM_C: break
        r_idx, c_idx = np.array(r_idx), np.array(c_idx)

    mapping = {int(our): int(model) for our, model in zip(r_idx.tolist(), c_idx.tolist())}

    def reorder_probs(p_model):
        p_new = np.zeros_like(p_model)
        for our in range(NUM_C):
            model_idx = mapping.get(our, our)  # identity if missing
            p_new[our] = p_model[model_idx]
        return p_new

    return mapping, reorder_probs

perm_resnet_map, resnet_reorder = infer_perm_on_subset("resnet50")
perm_vit_map,    vit_reorder    = infer_perm_on_subset("vit")
print("ResNet-50 index mapping (our_idx -> resnet_idx):", perm_resnet_map)
print("ViT       index mapping (our_idx -> vit_idx):    ", perm_vit_map)

# Per-image prediction (aligned) 
@torch.no_grad()
def predict_proba_all_models_aligned(pil_img: Image.Image) -> dict:
    outs = {}

    # CNN / DenseNet / ConvNeXt / MobileNetV3 share the same 224/0.5 pipeline
    x = preprocess_224_05(pil_img).unsqueeze(0).to(device)
    outs["cnn"]           = torch.softmax(cnn_model(x),       dim=1).cpu().numpy()[0]
    outs["densenet121"]   = torch.softmax(densenet_model(x),  dim=1).cpu().numpy()[0]
    outs["convnext_tiny"] = torch.softmax(convnext_model(x),  dim=1).cpu().numpy()[0]
    outs["mobilenet_v3"]  = torch.softmax(mobilenet_model(x), dim=1).cpu().numpy()[0]

    # ResNet-50 (ImageNet norm) + permutation
    x = preprocess_resnet(pil_img).unsqueeze(0).to(device)
    p_res = torch.softmax(resnet_model(x), dim=1).cpu().numpy()[0]
    outs["resnet50"] = resnet_reorder(p_res)

    # EfficientNet-B3 (300/0.5)
    x = preprocess_effnet(pil_img).unsqueeze(0).to(device)
    outs["efficientnet_b3"] = torch.softmax(effnet_model(x), dim=1).cpu().numpy()[0]

    # ViT + permutation
    x = preprocess_vit(pil_img).unsqueeze(0).to(device)
    p_vit = torch.softmax(vit_model(x).logits, dim=1).cpu().numpy()[0]
    outs["vit"] = vit_reorder(p_vit)

    # Mean ensemble (aligned)
    stack = np.stack([outs[k] for k in outs.keys()], axis=0)  # M x C
    outs["ensemble_mean"] = stack.mean(axis=0)
    return outs

# Score on test_df 
model_keys      = ["cnn","resnet50","densenet121","efficientnet_b3","convnext_tiny","mobilenet_v3","vit"]
per_model_probs = {k: [] for k in model_keys}
ensemble_probs  = []
labels          = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Scoring models on test set"):
    p = row["path"]
    try:
        img = Image.open(p).convert("RGB")
    except Exception:
        continue
    outs = predict_proba_all_models_aligned(img)
    for k in model_keys:
        per_model_probs[k].append(outs[k])
    ensemble_probs.append(outs["ensemble_mean"])
    labels.append(int(row["label"]))

labels = np.array(labels)
for k in model_keys:
    per_model_probs[k] = np.array(per_model_probs[k])
ensemble_probs = np.array(ensemble_probs)
print(f"Used {len(labels)} images out of {len(test_df)}.")

# Build comparison table 
def metrics_from_probs(y_true, p):
    preds = p.argmax(axis=1)
    acc = accuracy_score(y_true, preds)
    f1  = f1_score(y_true, preds, average="weighted")
    try:
        aucM = roc_auc_score(y_true, p, multi_class="ovr", average="macro")
        aucm = roc_auc_score(y_true, p, multi_class="ovr", average="micro")
    except ValueError:
        aucM, aucm = np.nan, np.nan
    return acc, f1, aucM, aucm

def count_params(m):
    return sum(p.numel() for p in m.parameters())/1e6

param_count = {
    "cnn":             count_params(cnn_model),
    "resnet50":        count_params(resnet_model),
    "densenet121":     count_params(densenet_model),
    "efficientnet_b3": count_params(effnet_model),
    "convnext_tiny":   count_params(convnext_model),
    "mobilenet_v3":    count_params(mobilenet_model),
    "vit":             count_params(vit_model),
    "ensemble_mean":   np.nan
}

rows = []
for k in model_keys:
    acc, f1, aucM, aucm = metrics_from_probs(labels, per_model_probs[k])
    rows.append({
        "Model": k,
        "Params (M)": round(param_count[k], 3),
        "Accuracy": round(acc, 4),
        "F1 (weighted)": round(f1, 4),
        "AUC (macro, OvR)": round(aucM, 4) if not np.isnan(aucM) else np.nan,
        "AUC (micro, OvR)": round(aucm, 4) if not np.isnan(aucm) else np.nan
    })

acc_e, f1_e, aucM_e, aucm_e = metrics_from_probs(labels, ensemble_probs)
rows.append({
    "Model": "ensemble_mean",
    "Params (M)": np.nan,
    "Accuracy": round(acc_e, 4),
    "F1 (weighted)": round(f1_e, 4),
    "AUC (macro, OvR)": round(aucM_e, 4) if not np.isnan(aucM_e) else np.nan,
    "AUC (micro, OvR)": round(aucm_e, 4) if not np.isnan(aucm_e) else np.nan
})

comparison_df = pd.DataFrame(rows).sort_values(
    by=["Accuracy", "F1 (weighted)"], ascending=False
).reset_index(drop=True)

display(comparison_df)
out_csv = "/kaggle/working/model_comparison.csv"
comparison_df.to_csv(out_csv, index=False)
print(f"Saved: {out_csv}")

**Table 1.** Performance comparison of all models (CNN, ResNet-50, ViT, DenseNet-121, EfficientNet-B3, ConvNeXt-Tiny, MobileNetV3) and the ensemble on the HAM10000 test set. Metrics reported: Accuracy, Weighted F1, Macro ROC–AUC, and Micro ROC–AUC.  

### Model Performance Comparison — Accuracy/F1 and ROC–AUC

In [ ]:
# Model comparison charts (Accuracy/F1 and AUCs)
assert 'comparison_df' in globals(), "Run the comparison table cell first."

# Pretty names & ordering
pretty = {
    "cnn": "CNN",
    "resnet50": "ResNet-50",
    "densenet121": "DenseNet-121",
    "efficientnet_b3": "EfficientNet-B3",
    "convnext_tiny": "ConvNeXt-Tiny",
    "mobilenet_v3": "MobileNetV3",
    "vit": "ViT",
    "ensemble_mean": "Ensemble"
}
order = ["cnn","resnet50","densenet121","efficientnet_b3",
         "convnext_tiny","mobilenet_v3","vit","ensemble_mean"]

df = comparison_df.copy()
df['Model'] = df['Model'].map(pretty).fillna(df['Model'])
present = [pretty.get(m, m) for m in order if pretty.get(m, m) in df['Model'].values]
df = df.set_index('Model').loc[present].reset_index()

# Global style 
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 16,
    "axes.labelsize": 12,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "axes.facecolor": "white",
})

# Helpers
def _pct_fmt(x, _): return f"{x:.0%}"

def _annotate(ax, bars, decimals=2):
    """Label bars with values; small white tag for readability."""
    for b in bars:
        h = b.get_height()
        ax.annotate(f"{h:.{decimals}f}",
                    xy=(b.get_x() + b.get_width()/2., h),
                    xytext=(0, 4), textcoords="offset points",
                    ha="center", va="bottom", fontsize=9,
                    bbox=dict(boxstyle="round,pad=0.15",
                              facecolor="white", alpha=0.85, linewidth=0))

def _finish(ax, title, ylab, ymin, ymax):
    ax.set_title(title, pad=10, weight="semibold")
    ax.set_ylabel(ylab)
    ax.yaxis.set_major_formatter(FuncFormatter(_pct_fmt))
    ax.set_ylim(ymin, ymax)
    ax.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.35)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)

def _accent_ensemble(bars, labels):
    """Hatch & outline Ensemble bars only (legend remains clean)."""
    if "Ensemble" in labels:
        i = labels.index("Ensemble")
        bars[i].set_linewidth(1.6)
        bars[i].set_edgecolor("black")
        bars[i].set_hatch("//")
        bars[i].set_alpha(0.98)

x = np.arange(len(df))
width = 0.34
blue, orange = "#1f77b4", "#ff7f0e"

# Figure 1 — Accuracy vs F1
fig1, ax1 = plt.subplots(figsize=(11.2, 4.6), constrained_layout=True)

b1 = ax1.bar(x - width/2, df["Accuracy"].values,     width, color=blue,   label="Accuracy")
b2 = ax1.bar(x + width/2, df["F1 (weighted)"].values, width, color=orange, label="F1 (weighted)")

_accent_ensemble(b1, df["Model"].tolist())
_accent_ensemble(b2, df["Model"].tolist())

ax1.set_xticks(x, df["Model"].values, rotation=14, ha="right")
# Start at 60% to show separation, cap a little above max
upper1 = max(1.0, float(np.nanmax(df[["Accuracy","F1 (weighted)"]])) + 0.02)
_finish(ax1, "Model Comparison — Accuracy vs Weighted F1", "Score", ymin=0.60, ymax=upper1)
_annotate(ax1, b1, decimals=2)
_annotate(ax1, b2, decimals=2)

# Clean legend (no hatch)
ax1.legend(handles=[
    Patch(facecolor=blue,   edgecolor="none", label="Accuracy"),
    Patch(facecolor=orange, edgecolor="none", label="F1 (weighted)"),
], loc="upper left", frameon=False, ncol=2)

p1_png = "/kaggle/working/model_comp_accuracy_f1.png"
p1_svg = "/kaggle/working/model_comp_accuracy_f1.svg"
fig1.savefig(p1_png, bbox_inches="tight")
fig1.savefig(p1_svg, bbox_inches="tight")
print(f"Saved: {p1_png}\nSaved: {p1_svg}")

# Figure 2 — ROC–AUC (Macro & Micro, One-vs-Rest)
metrics2 = ["AUC (macro, OvR)", "AUC (micro, OvR)"]
x = np.arange(len(df))
width = 0.32                     # a bit narrower for breathing room
blue, orange = "#1f77b4", "#ff7f0e"

fig2, ax2 = plt.subplots(figsize=(11.2, 4.6), constrained_layout=True)

b3 = ax2.bar(x - width/2, df[metrics2[0]].values, width, color=blue,   label="AUC (Macro, OvR)")
b4 = ax2.bar(x + width/2, df[metrics2[1]].values, width, color=orange, label="AUC (Micro, OvR)")

_accent_ensemble(b3, df["Model"].tolist())
_accent_ensemble(b4, df["Model"].tolist())

ax2.set_xticks(x, df["Model"].values, rotation=14, ha="right")
_finish(ax2, "Model Comparison — ROC–AUC (Macro & Micro, OvR)", "ROC–AUC",
        ymin=0.90, ymax=1.005)

_annotate(ax2, b3, decimals=2)
_annotate(ax2, b4, decimals=2)

# Legend above the plot, clean & centered
legend_handles = [
    Patch(facecolor=blue,   edgecolor="none", label="AUC (Macro, OvR)"),
    Patch(facecolor=orange, edgecolor="none", label="AUC (Micro, OvR)")
]
leg = ax2.legend(handles=legend_handles,
                 loc="upper center", bbox_to_anchor=(0.5, 1.18),
                 ncol=2, frameon=False, borderaxespad=0.)

fig2.savefig("/kaggle/working/model_comp_auc.png", bbox_inches="tight")
fig2.savefig("/kaggle/working/model_comp_auc.svg", bbox_inches="tight")
plt.show()

**Figure 1.** Model comparison of Accuracy and Weighted F1 for all models and the ensemble. Ensemble highlighted with hatch and outline.  

**Figure 2.** Model comparison of ROC–AUC (Macro and Micro, OvR) for all models and the ensemble. Ensemble highlighted with hatch and outline.  

## 3.2 Grad-CAM Explainability

###  Define Custom CNN Model

In [ ]:
class CNNModel(nn.Module):
    def __init__(self, num_classes=7):
        super(CNNModel, self).__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        # 224x224 -> after 4 pools -> 14x14 feature map with 256 channels
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

### Load Trained Models for Ensemble Grad-CAM

In [ ]:
# CNN (Custom)
cnn_model = CNNModel()
cnn_model.load_state_dict(torch.load(
    "/kaggle/input/kaggleinputham10000-model-weights/cnn_ham10000_best_model.pth",
    map_location=device
))
cnn_model.to(device)
cnn_model.eval()

# ResNet50 (TorchVision)
resnet_model = torchvision.models.resnet50(pretrained=False, num_classes=7)
resnet_model.load_state_dict(torch.load(
    "/kaggle/input/resnet50-ham10000-model-weights/resnet50_ham10000.pth",
    map_location=device
))
resnet_model.to(device)
resnet_model.eval()

# DenseNet121 (TorchVision)
densenet_model = torchvision.models.densenet121(pretrained=False, num_classes=7)
densenet_model.load_state_dict(torch.load(
    "/kaggle/input/densenet121-ham10000-model-weights/densenet121_ham10000.pth",
    map_location=device
))
densenet_model.to(device)
densenet_model.eval()

# EfficientNet B3 (TorchVision)
effnet_model = efficientnet_b3(pretrained=False)
effnet_model.classifier[1] = nn.Linear(effnet_model.classifier[1].in_features, 7)
effnet_model.load_state_dict(torch.load(
    "/kaggle/input/efficientnetb3-ham10000-model-weights/efficientnet_b3_best.pth",
    map_location=device
))
effnet_model.to(device)
effnet_model.eval()

# ViT (Hugging Face, manually initialized)
vit_config = ViTConfig.from_pretrained("google/vit-base-patch16-224")
vit_config.num_labels = 7

vit_model = ViTForImageClassification(vit_config)
vit_model.load_state_dict(torch.load(
    "/kaggle/input/vit-ham10000-model-weights/vit_ham10000_best_model.pth",
    map_location=device
))
vit_model.to(device)
vit_model.eval()

# ConvNeXt Tiny (TorchVision)
convnext_model = convnext_tiny(num_classes=7)
convnext_model.load_state_dict(torch.load(
    "/kaggle/input/ham10000-convnext-tiny-weights/convnext_tiny_ham10000.pth",
    map_location=device
))
convnext_model.to(device)
convnext_model.eval()

# MobileNet V3 (TorchVision)
mobilenet_model = mobilenet_v3_large(num_classes=7)
mobilenet_model.load_state_dict(torch.load(
    "/kaggle/input/mobilenetv3-ham10000-model-weights/mobilenetv3_ham10000.pth",
    map_location=device
))
mobilenet_model.to(device)
mobilenet_model.eval()

### Define ViT Wrapper for Grad-CAM Compatibility

In [ ]:
class WrappedViT(nn.Module):
    def __init__(self, vit_model):
        super().__init__()
        self.vit = vit_model.vit
        self.classifier = vit_model.classifier

    def forward(self, x):
        outputs = self.vit(pixel_values=x)
        logits = self.classifier(outputs.last_hidden_state[:, 0])
        return logits

### Define Preprocessing Transforms

In [ ]:
preprocess_224_05 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

preprocess_resnet = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

preprocess_effnet = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

preprocess_vit = preprocess_224_05

### Load Pretrained Models

In [ ]:
# Load pretrained ViT model (or fine-tuned one if saved a .pth)
vit_model = ViTForImageClassification.from_pretrained("google/vit-base-patch16-224")
vit_model.to(device)
vit_model.eval()

###  Initialize Wrapped ViT + Grad-CAM Configurations

In [ ]:
wrapped_vit = WrappedViT(vit_model).to(device)

gradcam_configs = {
    "cnn":            (cnn_model, cnn_model.features[-3], preprocess_224_05),
    "resnet50":       (resnet_model, resnet_model.layer4[-1], preprocess_resnet),
    "densenet121":    (densenet_model, densenet_model.features[-1], preprocess_224_05),
    "efficientnet_b3":(effnet_model, effnet_model.features[-1], preprocess_effnet),
    "convnext_tiny":  (convnext_model, convnext_model.features[-1], preprocess_224_05),
    "mobilenet_v3":   (mobilenet_model, mobilenet_model.features[-1], preprocess_224_05),

    # Correct ViT layer (projection returns 4D output: [B, C, H, W])
    "vit":            (wrapped_vit, vit_model.vit.embeddings.patch_embeddings.projection, preprocess_vit),
}

### Define Grad-CAM Utility Functions

In [ ]:
def get_cam_for_model(model, target_layer, preprocess_fn, pil_img, class_idx):
    input_tensor = preprocess_fn(pil_img).unsqueeze(0).to(device)
    cam = GradCAM(model=model, target_layers=[target_layer])
    grayscale_cam = cam(input_tensor=input_tensor,
                        targets=[ClassifierOutputTarget(class_idx)],
                        eigen_smooth=True)[0]

    rgb_img = np.array(pil_img.resize(grayscale_cam.shape[::-1])) / 255.0
    vis_img = show_cam_on_image(rgb_img.astype(np.float32), grayscale_cam, use_rgb=True)
    return vis_img, grayscale_cam


def get_ensemble_gradcam(pil_img, class_idx):
    cams = []
    target_size = None  # Will use first cam's size as reference

    for key in gradcam_configs:
        model, target_layer, preprocess_fn = gradcam_configs[key]
        try:
            _, cam = get_cam_for_model(model, target_layer, preprocess_fn, pil_img, class_idx)

            # Store the shape from the first successful cam
            if target_size is None:
                target_size = cam.shape[::-1]  # (width, height)

            # Resize the cam to the target size
            cam_resized = cv2.resize(cam, target_size)
            cams.append(cam_resized)
        except Exception as e:
            print(f"Failed on {key}: {e}")

    if not cams:
        return None

    # Average all grayscale CAMs
    avg_cam = np.mean(np.stack(cams, axis=0), axis=0)

    # Blend with original image
    rgb_img = np.array(pil_img.resize(avg_cam.shape[::-1])) / 255.0
    vis_img = show_cam_on_image(rgb_img.astype(np.float32), avg_cam, use_rgb=True)

    return vis_img.astype(np.uint8)

### Load HAM10000 Metadata and Sample 20% Test Set

In [ ]:
# Load full metadata (10,015 images)
metadata_df = pd.read_csv("/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_metadata.csv")

# Use 20% as test set (~2003 samples)
sample_df = metadata_df.sample(frac=0.2, random_state=42).reset_index(drop=True)

# Rename 'dx' to 'label' if needed for consistency
sample_df = sample_df.rename(columns={"dx": "label"})

###  Map Image Paths

In [ ]:
def get_image_path(image_id):
    if f"{image_id}.jpg" in os.listdir("/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1"):
        return f"/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1/{image_id}.jpg"
    else:
        return f"/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2/{image_id}.jpg"

sample_df["path"] = sample_df["image_id"].apply(get_image_path)

### Save Sampled Test Set to CSV

In [ ]:
# Save to CSV
sample_df.to_csv("/kaggle/working/test_2003.csv", index=False)

# Check if file exists
if os.path.exists("/kaggle/working/test_2003.csv"):
    print("File saved successfully!")

    # Preview first 5 rows
    preview_df = pd.read_csv("/kaggle/working/test_2003.csv")
    print("Preview:")
    print(preview_df.head())
else:
    print("File not found. Something went wrong.")

### Define Class Names

In [ ]:
class_names = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']

### Generate Ensemble Grad-CAM Visualizations

In [ ]:
os.makedirs("/kaggle/working/gradcam_ensemble", exist_ok=True)

for i, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    try:
        pil_img = Image.open(row["path"]).convert("RGB")
        for class_idx in range(7):  # One heatmap per class
            vis = get_ensemble_gradcam(pil_img, class_idx=class_idx)
            if vis is not None:
                filename = f"{i}_{class_names[class_idx]}.jpg"
                save_path = os.path.join("/kaggle/working/gradcam_ensemble", filename)
                Image.fromarray(vis).save(save_path)
    except Exception as e:
        print(f"Failed on {row['path']}: {e}")

### Zip Grad-CAM Results

In [ ]:
gradcam_dir = "/kaggle/working/gradcam_ensemble"
if os.path.exists(gradcam_dir) and len(os.listdir(gradcam_dir)) > 0:
    zip_path = shutil.make_archive(gradcam_dir, 'zip', gradcam_dir)
    print(f"Zipped successfully: {zip_path}")
else:
    print("Directory is still empty. Debug Grad-CAM generation first.")

### Visualize Class Distribution

In [ ]:
sample_df["label"].value_counts()

## 3.3 ISIC External Evaluation

This section evaluates the generalization performance of all trained models—CNN, ResNet-50, DenseNet-121, EfficientNet-B3, ConvNeXt-Tiny, MobileNetV3, and ViT—on the external ISIC 2019 dataset. To ensure label compatibility with HAM10000, the ISIC classes AK and SCC were merged into a unified `akiec` category, and only the seven overlapping diagnostic classes were retained.

The external evaluation pipeline includes the following components:

- **Model-wise Inference:** Each model predicts probabilities on the ISIC 2019 dataset, enabling comparative performance analysis.
- **Classification Metrics:** Accuracy, macro and weighted F1-score, and macro ROC-AUC are computed for each model to assess predictive robustness under domain shift.
- **Confusion Matrices and ROC Curves:** Visual performance diagnostics are generated per model and per class.
- **Calibration Analysis:** Expected Calibration Error (ECE) is calculated for each model and the ensemble, offering insight into the reliability of predicted confidence scores.
- **Ensemble Performance:** A soft-voting ensemble is evaluated on the ISIC dataset to assess its effectiveness in generalization beyond the training domain.
- **Visual Explainability:** Grad-CAM heatmaps are computed for ensemble predictions on the ISIC 2019 set to visualize attention and support clinician interpretability.

All evaluation outputs—including metric tables, ROC plots, Grad-CAM visualizations, and exportable `.csv`/`.tex` summaries—are generated for reproducibility and publication.

This external validation provides a robust, end-to-end benchmark for assessing model transferability and clinical viability in dermatology AI systems.

### Constants and Class Mappings

In [ ]:
ham10000_classes = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
class_to_idx = {cls: i for i, cls in enumerate(ham10000_classes)}

# ISIC to HAM10000 label mapping
label_map = {
    0: 'nv', 1: 'mel', 2: 'bkl', 3: 'bcc', 4: 'ak', 5: 'vasc', 6: 'df', 7: 'scc'
}
isic_map = {
    'nv': 'nv', 'mel': 'mel', 'bkl': 'bkl', 'bcc': 'bcc',
    'ak': 'akiec', 'scc': 'akiec',
    'vasc': 'vasc', 'df': 'df'
}

ISIC_CSV = "/kaggle/input/isic-2019-jpg-224x224-resized/train-metadata.csv"
ISIC_IMG = "/kaggle/input/isic-2019-jpg-224x224-resized/train-image/image"

### Prepare ISIC Dataset DataFrame

In [ ]:
df = pd.read_csv(ISIC_CSV)

df["diagnosis"] = df["target"].map(label_map)
df["diagnosis"] = df["diagnosis"].map(isic_map)
df = df[df["diagnosis"].isin(ham10000_classes)].reset_index(drop=True)
df["label"] = df["diagnosis"].map(class_to_idx)

df["img_path"] = df["isic_id"].apply(lambda x: os.path.join(ISIC_IMG, f"{x}.jpg"))
df = df[df["img_path"].apply(os.path.exists)]

print(f"Valid ISIC Samples: {len(df)}")
print(df[['isic_id', 'diagnosis', 'label']].head())

### Load Models with Weights

In [ ]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 7

# Custom CNN
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=7):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Instantiate models
cnn_model = SimpleCNN(num_classes)

resnet_model = resnet50(pretrained=False)
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, num_classes)

densenet_model = densenet121(pretrained=False)
densenet_model.classifier = nn.Linear(densenet_model.classifier.in_features, num_classes)

effnet_model = efficientnet_b3(weights=None)
effnet_model.classifier[1] = nn.Linear(effnet_model.classifier[1].in_features, num_classes)

convnext_model = convnext_tiny(weights=None)
convnext_model.classifier[2] = nn.Linear(convnext_model.classifier[2].in_features, num_classes)

mobilenet_model = mobilenet_v3_large(pretrained=False)
mobilenet_model.classifier[3] = nn.Linear(mobilenet_model.classifier[3].in_features, num_classes)

# ViT from Hugging Face (not timm)
vit_config = ViTConfig.from_pretrained("/kaggle/input/vit-ham10000-model-weights")
vit_config.num_labels = num_classes
vit_model = ViTForImageClassification(vit_config)
vit_model.load_state_dict(torch.load(
    "/kaggle/input/vit-ham10000-model-weights/vit_ham10000_best_model.pth",
    map_location=device
))
vit_model = vit_model.to(device)
vit_model.eval()

# Load weights for CNN and CNN-based models
cnn_model.load_state_dict(torch.load(
    "/kaggle/input/kaggleinputham10000-model-weights/cnn_ham10000_best_model.pth",
    map_location=device
))
resnet_model.load_state_dict(torch.load(
    "/kaggle/input/resnet50-ham10000-model-weights/resnet50_ham10000.pth",
    map_location=device
))
densenet_model.load_state_dict(torch.load(
    "/kaggle/input/densenet121-ham10000-model-weights/densenet121_ham10000.pth",
    map_location=device
))
effnet_model.load_state_dict(torch.load(
    "/kaggle/input/efficientnetb3-ham10000-model-weights/efficientnet_b3_best.pth",
    map_location=device
))
convnext_model.load_state_dict(torch.load(
    "/kaggle/input/ham10000-convnext-tiny-weights/convnext_tiny_ham10000.pth",
    map_location=device
))
mobilenet_model.load_state_dict(torch.load(
    "/kaggle/input/mobilenetv3-ham10000-model-weights/mobilenetv3_ham10000.pth",
    map_location=device
))

# Bundle models
models_dict = {
    "CNN": cnn_model,
    "ResNet50": resnet_model,
    "DenseNet121": densenet_model,
    "EfficientNetB3": effnet_model,
    "ConvNeXtTiny": convnext_model,
    "MobileNetV3": mobilenet_model,
    "ViT": vit_model
}

# Move to device & set to eval
for name, model in models_dict.items():
    model.to(device)
    model.eval()

print(" All 7 models loaded with weights and moved to device. Ready for ISIC evaluation.")

### Load and Map ISIC Ground Truth to HAM10000 7-Class Labels

In [ ]:
# Load the uploaded ground truth CSV
gt_path = "/kaggle/input/isic-2019-groundtruth-7classes-ham10000-mapped/ISIC_2019_Training_GroundTruth.csv"
gt_df = pd.read_csv(gt_path)

# Diagnosis one-hot columns
onehot_cols = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']

# Derive single-label diagnosis if not already there
if 'diagnosis' not in gt_df.columns:
    gt_df["diagnosis"] = gt_df[onehot_cols].idxmax(axis=1).str.lower()
    gt_df["diagnosis"] = gt_df["diagnosis"].replace({"ak": "akiec", "scc": "akiec"})

# Add label encoding for HAM-compatible classes
ham_classes = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
class_to_idx = {cls: idx for idx, cls in enumerate(ham_classes)}
gt_df["label"] = gt_df["diagnosis"].map(class_to_idx)

# Add image path from 'image' column
image_dir = "/kaggle/input/isic-2019-jpg-224x224-resized/train-image/image"  # note the nested folder
gt_df["img_path"] = gt_df["image"].apply(lambda x: os.path.join(image_dir, f"{x}.jpg"))

# Filter only rows with valid image files
gt_df = gt_df[gt_df["img_path"].apply(os.path.exists)].reset_index(drop=True)

# Final check
print("Final ISIC 2019 test samples:", len(gt_df))
print("Label distribution:\n", gt_df["diagnosis"].value_counts())
print(gt_df.head())

### ISIC2019 Dataset Definition

In [ ]:
# Define the test-time transforms (resize, center crop, normalization)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),  # same as training
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # standard 3-channel
])

class ISIC2019Dataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["img_path"]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = row["label"]
        return image, label   # only two items

### Create DataLoader

In [ ]:
test_dataset = ISIC2019Dataset(gt_df, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# Quick check
for images, labels in test_loader:
    print(images.shape, labels.shape)
    break

### Inference Loop & Predictions

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

results = {}

for name, model in models_dict.items():
    print(f"\nEvaluating {name}...")
    model.to(device)
    model.eval()

    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc=f"{name} Inference"):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            # Handle Hugging Face ViT output
            if isinstance(outputs, dict) or hasattr(outputs, "logits"):
                outputs = outputs.logits
            probs = torch.softmax(outputs, dim=1)
            preds = probs.argmax(1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    results[name] = {
        "preds": np.array(all_preds),
        "labels": np.array(all_labels),
        "probs": np.array(all_probs)
    }

print("Inference complete for all models.")

### Metrics and Classification Reports

In [ ]:
class_names = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']

for name, res in results.items():
    preds = res["preds"]
    labels = res["labels"]
    probs = res["probs"]

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    try:
        auc = roc_auc_score(labels, probs, multi_class="ovr")
    except:
        auc = "N/A"

    present_classes = sorted(np.unique(np.concatenate((preds, labels))))
    present_class_names = [class_names[i] for i in present_classes]

    print(f"\n=== {name} ===")
    print(f"Accuracy: {acc:.4f}")
    print(f"Weighted F1: {f1:.4f}")
    print(f"ROC-AUC: {auc}")
    print("\nClassification Report:")
    print(classification_report(labels, preds, target_names=present_class_names, labels=present_classes))

### Save Results to JSON File

In [ ]:
# Convert NumPy arrays to lists for JSON compatibility
json_ready_results = {
    model: {
        "preds": res["preds"].tolist(),
        "labels": res["labels"].tolist(),
        "probs": res["probs"].tolist()
    }
    for model, res in results.items()
}

with open("/kaggle/working/isic2019_external_eval_results.json", "w") as f:
    json.dump(json_ready_results, f)

print("Saved model-wise predictions to: /kaggle/working/isic2019_external_eval_results.json")

### Save Metrics Summary to CSV

In [ ]:
summary_data = []

for name, res in results.items():
    preds = res["preds"]
    labels = res["labels"]
    probs = res["probs"]

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    try:
        auc = roc_auc_score(labels, probs, multi_class="ovr")
    except:
        auc = None

    summary_data.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Weighted_F1": round(f1, 4),
        "ROC_AUC": round(auc, 4) if auc else "N/A"
    })

# Save to CSV
summary_df = pd.DataFrame(summary_data)
summary_df.to_csv("/kaggle/working/isic2019_external_metrics_summary.csv", index=False)

print("Saved metrics summary to: /kaggle/working/isic2019_external_metrics_summary.csv")

### Save Classification Reports

In [ ]:
for name, res in results.items():
    report = classification_report(
        res["labels"], res["preds"],
        target_names=class_names,
        digits=4
    )
    with open(f"/kaggle/working/{name}_classification_report.txt", "w") as f:
        f.write(report)

print("Saved all classification reports to /kaggle/working/")

### Confusion Matrix Heatmaps

In [ ]:
for name, res in results.items():
    y_true = res["labels"]
    y_pred = res["preds"]

    # Compute confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)

    plt.title(f"Confusion Matrix — {name}")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.savefig(f"/kaggle/working/confusion_matrix_{name.lower()}.png", dpi=300)
    plt.show()

### Compute Ensemble Predictions on ISIC 2019

In [ ]:
# Compute ensemble predictions
ensemble_probs = np.mean([results[m]["probs"] for m in results.keys()], axis=0)
ensemble_preds = np.argmax(ensemble_probs, axis=1)

# All models use the same test set, so take labels from any one of them
ensemble_labels = results["CNN"]["labels"]

print("Ensemble predictions and labels computed.")

### Confusion Matrix for Ensemble Model on ISIC 2019

In [ ]:
# Use ensemble predictions
cm_ensemble = confusion_matrix(ensemble_labels, ensemble_preds, labels=range(len(class_names)))

plt.figure(figsize=(8, 6))
sns.heatmap(cm_ensemble, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)

plt.title("Confusion Matrix — Ensemble (ISIC 2019)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()

# Save outputs
plt.savefig("/kaggle/working/confusion_matrix_ensemble_isic2019.png", dpi=300)
plt.show()

**Figure 5:** Confusion matrix of the ensemble model on the ISIC 2019 dataset, showing strong classification performance across most classes with moderate misclassifications in minority categories.

### ROC Curve Per Class (Multi-Class ROC Plot)

In [ ]:
for name, res in results.items():
    y_true = res["labels"]
    y_score = res["probs"]

    y_true_bin = label_binarize(y_true, classes=range(len(class_names)))

    fpr = dict()
    tpr = dict()
    roc_auc = dict()

    plt.figure(figsize=(10, 8))
    for i in range(len(class_names)):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_score[:, i])
        roc_auc[i] = sk_auc(fpr[i], tpr[i])
        plt.plot(fpr[i], tpr[i], label=f"{class_names[i]} (AUC = {roc_auc[i]:.2f})")

    plt.plot([0, 1], [0, 1], 'k--', label='Chance')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve — {name}")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(f"/kaggle/working/roc_curve_{name.lower()}.png", dpi=300)
    plt.show()

### Generate and Save Ensemble ROC Curve for ISIC 2019

In [ ]:
# Generate ROC curve for Ensemble on ISIC 2019 
ensemble_name = "Ensemble"

# Compute ensemble probabilities by averaging logits or probabilities
ensemble_probs = np.mean([results[m]["probs"] for m in results.keys()], axis=0)
ensemble_preds = np.argmax(ensemble_probs, axis=1)
ensemble_labels = results["CNN"]["labels"]  # all models used same test set

# Compute class-wise ROC
y_true_bin = label_binarize(ensemble_labels, classes=range(len(class_names)))

plt.figure(figsize=(10, 8))
for i, cls in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], ensemble_probs[:, i])
    auc_val = sk_auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{cls} (AUC = {auc_val:.2f})")

plt.plot([0, 1], [0, 1], 'k--', label='Chance')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Ensemble (ISIC 2019)")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("/kaggle/working/roc_curve_ensemble_isic2019.png", dpi=300)
plt.show()

**Figure 6.** Class-wise ROC curves of the ensemble model on the ISIC 2019 dataset. The ensemble achieves consistently high AUCs for major lesion classes while showing slightly reduced discriminative power for minority categories.

### Save All Evaluation Metrics to CSV

In [ ]:
metrics_summary = []

for name, res in results.items():
    acc = accuracy_score(res["labels"], res["preds"])
    f1 = f1_score(res["labels"], res["preds"], average="weighted")
    try:
        auc_score = roc_auc_score(res["labels"], res["probs"], multi_class="ovr")
    except:
        auc_score = np.nan

    metrics_summary.append({
        "Model": name,
        "Accuracy": acc,
        "Weighted F1": f1,
        "ROC-AUC": auc_score
    })

metrics_df = pd.DataFrame(metrics_summary)
metrics_df.to_csv("/kaggle/working/isic2019_model_metrics.csv", index=False)
metrics_df

**Table 2.** Performance metrics (Accuracy, F1-score, AUC) of all models on the external ISIC 2019 dataset. Results reflect generalization under domain shift from HAM10000 training.

### Save All Predictions to CSV

In [ ]:
final_df = gt_df.copy()

for name, res in results.items():
    final_df[f"{name}_pred"] = res["preds"]
    final_df[f"{name}_probs"] = res["probs"].tolist()  # saves list of 7 prob values per row

final_df.to_csv("/kaggle/working/isic2019_all_model_predictions.csv", index=False)

### Rank Models by Accuracy, F1, AUC

In [ ]:
# Collect all metrics
metrics_df = []

for name, res in results.items():
    acc = accuracy_score(res["labels"], res["preds"])
    f1 = f1_score(res["labels"], res["preds"], average="weighted")
    try:
        auc_score = roc_auc_score(res["labels"], res["probs"], multi_class="ovr")
    except:
        auc_score = np.nan
    metrics_df.append({"Model": name, "Accuracy": acc, "F1": f1, "AUC": auc_score})

metrics_df = pd.DataFrame(metrics_df)
metrics_df["MeanRank"] = metrics_df[["Accuracy", "F1", "AUC"]].rank(ascending=False).mean(axis=1)
metrics_df = metrics_df.sort_values("MeanRank")

# Save to CSV
metrics_df.to_csv("/kaggle/working/model_ranking.csv", index=False)
metrics_df

### Zip /kaggle/working/ and Prepare for Download

In [ ]:
zip_path = "/kaggle/working/isic2019_outputs.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for foldername, subfolders, filenames in os.walk("/kaggle/working/"):
        for filename in filenames:
            if filename.endswith(".png") or filename.endswith(".csv"):
                filepath = os.path.join(foldername, filename)
                arcname = os.path.relpath(filepath, "/kaggle/working/")
                zipf.write(filepath, arcname)

print("All outputs zipped at:", zip_path)

### Aggregate the Model Metrics

In [ ]:
# Class names (if needed for later table usage)
class_names = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']

# Initialize list for all metrics
metrics_list = []

# Collect metrics
for name, res in results.items():
    preds = res["preds"]
    labels = res["labels"]
    probs = res["probs"]

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    
    try:
        auc = roc_auc_score(labels, probs, multi_class="ovr")
    except:
        auc = None

    metrics_list.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Weighted F1": round(f1, 4),
        "ROC-AUC": round(auc, 4) if auc else "N/A"
    })

# Create DataFrame
df_metrics = pd.DataFrame(metrics_list)
df_metrics_sorted = df_metrics.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)

### Export to LaTeX

In [ ]:
latex_table = df_metrics_sorted.to_latex(index=False, column_format="lccc", float_format="%.4f", caption="Model Performance on ISIC 2019 Dataset", label="tab:model-results")

# Save LaTeX table
with open("/kaggle/working/model_performance_table.tex", "w") as f:
    f.write(latex_table)

print("LaTeX table saved to: /kaggle/working/model_performance_table.tex")

### Export to Excel

In [ ]:
excel_path = "/kaggle/working/model_performance.xlsx"
df_metrics_sorted.to_excel(excel_path, index=False)

print(f"Excel table saved to: {excel_path}")

# 4. Results

## Results

The performance of all models and the ensemble was evaluated on the **HAM10000 test set** and externally validated on the **ISIC 2019 dataset**, using **Accuracy, F1-score, and ROC–AUC (macro and micro)**.

**Table 1** reports the performance comparison of **CNN, ResNet-50, ViT, DenseNet-121, EfficientNet-B3, ConvNeXt-Tiny, MobileNetV3**, and the **ensemble** on the **HAM10000** test set.  
The ensemble consistently achieved the best overall performance across all metrics.

**Figure 1** presents a bar-plot comparison of **Accuracy** and **Weighted F1** scores for all models.  
The ensemble is highlighted with hatch and outline, showing clear improvements over individual models.

**Figure 2** illustrates model comparisons for **ROC–AUC (Macro and Micro, OvR)**.  
Again, the ensemble outperforms all individual baselines.

**Figure 3** shows the **confusion matrix of ensemble predictions** on the HAM10000 test set, illustrating strong per-class classification with only limited misclassifications.

**Figure 4** provides the **ROC curves for the ensemble across all lesion classes on HAM10000**.  
Macro- and micro-averaged AUC values confirm robust performance.

To evaluate **generalization under domain shift**, we tested all models on the **ISIC 2019 dataset**:

**Table 2** summarizes classification metrics (**Accuracy, F1-score, AUC**) on **ISIC 2019**.  
Results reflect generalization performance beyond HAM10000 training.

**Figure 5:** Confusion matrix of the **ensemble model** on the **ISIC 2019 dataset**, showing strong classification performance across most classes with moderate misclassifications in minority categories.

**Figure 6:** Class-wise **ROC curves of the ensemble model on the ISIC 2019 dataset**.  
The ensemble achieves consistently high AUCs for major lesion classes while showing slightly reduced discriminative power for minority categories.

**Table 3** summarizes **McNemar’s test** results comparing the **ensemble** against each individual model on the **HAM10000 test set**.  
The ensemble achieved statistically significant improvements (*p* < 0.05) in most comparisons.

**Table 4** quantifies **performance deltas (Ensemble – Individual model)** with **95 % confidence intervals**, confirming consistent gains in Accuracy, F1, and AUC.

Finally, calibration was assessed using **reliability diagrams**.  
**Table 5** shows the **Expected Calibration Error (ECE)**, demonstrating that the ensemble maintains competitive calibration compared to individual models.

---

### Figures and Tables

- **Table 1.** Performance comparison of all models (CNN, ResNet-50, ViT, DenseNet-121, EfficientNet-B3, ConvNeXt-Tiny, MobileNetV3) and the ensemble on the **HAM10000 test set**.  
  Metrics reported: **Accuracy, Weighted F1, Macro ROC–AUC, and Micro ROC–AUC**.

- **Figure 1.** Model comparison of **Accuracy** and **Weighted F1** for all models and the ensemble.  
  Ensemble highlighted with hatch and outline.

- **Figure 2.** Model comparison of **ROC–AUC (Macro and Micro, OvR)** for all models and the ensemble.  
  Ensemble highlighted with hatch and outline.

- **Figure 3.** **Confusion matrix** of ensemble predictions on the **HAM10000 test set**, illustrating per-class classification performance.

- **Figure 4.** **ROC curves for the ensemble** across all lesion classes on **HAM10000**, with macro- and micro-AUC values reported.

- **Table 2.** **Performance metrics (Accuracy, F1-score, AUC)** of all models on the external **ISIC 2019 dataset**.  
  Results reflect generalization under domain shift from HAM10000 training.

- **Figure 5.** **Confusion matrix of the ensemble model on the ISIC 2019 dataset**, showing strong classification performance across most classes with moderate misclassifications in minority categories.

- **Figure 6.** **Class-wise ROC curves of the ensemble model on the ISIC 2019 dataset**.  
  The ensemble achieves consistently high AUCs for major lesion classes while showing slightly reduced discriminative power for minority categories.

- **Table 3.** **McNemar’s test** results comparing the **ensemble** against each individual model on the **HAM10000 test set**.  
  Reported are disagreement counts (b, c) and *p-values*.

- **Table 4.** **Performance deltas (Ensemble – Individual model)** with **95 % confidence intervals** for **Accuracy, Weighted F1, and ROC–AUC (macro)** on the **HAM10000 test set**.

- **Table 5.** **Reliability diagrams (Expected Calibration Error, ECE)** showing calibration of **ensemble predictions** versus ground-truth probabilities.

## 4.1 Ensemble Evaluation

### Full Test Set Evaluation — Individual Models & Ensemble

In [ ]:
# Aligned Ensemble on full test_balanced.csv 
assert 'class_names' in globals(), "Define class_names first."
assert 'device' in globals(), "Define device (cuda/cpu)."
NUM_C = len(class_names)

# Point this to the full test CSV
CANDIDATE_TEST_CSVS = [
    "/kaggle/working/test_2003.csv"  # final test set (20% of HAM10000)
]
test_csv_path = next((p for p in CANDIDATE_TEST_CSVS if os.path.exists(p)), None)
assert test_csv_path, f"Could not find test_balanced.csv in: {CANDIDATE_TEST_CSVS}"

# HAM10000 image folders (read-only)
IMG_DIRS = [
    "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1",
    "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2",
]

def find_image_path_from_id(image_id: str):
    fname = f"{image_id}.jpg"
    for d in IMG_DIRS:
        p = os.path.join(d, fname)
        if os.path.exists(p):
            return p
    return None

# Preprocess pipelines (match with training for each model family) 
warnings.filterwarnings("ignore", category=UserWarning)
to_tensor = transforms.ToTensor()
resize_224 = transforms.Resize((224, 224))
resize_300 = transforms.Resize((300, 300))
normalize_05 = transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
normalize_imagenet = transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])

preprocess_224_05  = transforms.Compose([resize_224, to_tensor, normalize_05])        # CNN / DenseNet / ConvNeXt / MobileNet
preprocess_resnet  = transforms.Compose([resize_224, to_tensor, normalize_imagenet])  # ResNet-50
preprocess_effnet  = transforms.Compose([resize_300, to_tensor, normalize_05])        # EfficientNet-B3

vit_processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")
def preprocess_vit(pil_img: Image.Image) -> torch.Tensor:
    enc = vit_processor(images=pil_img, return_tensors="pt")
    return enc["pixel_values"].squeeze(0)  # CHW

# Load full test df and ensure 'path' exists 
test_df = pd.read_csv(test_csv_path)
if "path" not in test_df.columns or test_df["path"].isna().any():
    assert "image_id" in test_df.columns, "test_balanced.csv must have either 'path' or 'image_id'."
    test_df["path"] = test_df["image_id"].apply(find_image_path_from_id)

# Ensure labels are numeric class indices
if test_df["label"].dtype == "object":
    label2id = {name: idx for idx, name in enumerate(class_names)}
    test_df["label"] = test_df["label"].map(label2id)

# Build a subset to infer class-index permutations for ResNet-50 & ViT 
def _valid_rows(df):
    rows = []
    for _, r in df.iterrows():
        y = int(r["label"])
        p = r["path"]
        if os.path.exists(p):
            rows.append((y, p))
    return rows

pool = _valid_rows(test_df)
sample_for_perm = pool[: min(800, len(pool))]  # up to 800 examples is plenty

def infer_perm_on_subset(model_name):
    """Return (mapping_dict, reorder_fn) so model probs map into our class order."""
    counts = np.zeros((NUM_C, NUM_C), dtype=np.int64)  # rows=true(our), cols=pred(model)

    for (y_true, path) in tqdm(sample_for_perm, desc=f"Inferring perm for {model_name}", leave=False):
        img = Image.open(path).convert("RGB")

        if model_name == "resnet50":
            x = preprocess_resnet(img).unsqueeze(0).to(device)
            with torch.no_grad(): logits = resnet_model(x)
            pred = int(logits.argmax(1).item())

        elif model_name == "vit":
            x = preprocess_vit(img).unsqueeze(0).to(device)
            with torch.no_grad(): logits = vit_model(x).logits
            pred = int(logits.argmax(1).item())

        else:
            raise ValueError("Only use this for resnet50 or vit.")

        if 0 <= y_true < NUM_C and 0 <= pred < NUM_C:
            counts[y_true, pred] += 1

    try:
        r_idx, c_idx = linear_sum_assignment(-counts)  # maximize diagonal
    except Exception:
        # greedy fallback
        used_true = set(); r_idx=[]; c_idx=[]
        col_order = counts.argmax(axis=0)
        pairs = sorted([(counts[r,c], r, c) for c, r in enumerate(col_order)], reverse=True)
        for _, r, c in pairs:
            if r not in used_true:
                r_idx.append(r); c_idx.append(c); used_true.add(r)
            if len(used_true) == NUM_C: break
        r_idx = np.array(r_idx); c_idx = np.array(c_idx)

    mapping = {our: model for (our, model) in zip(r_idx.tolist(), c_idx.tolist())}

    def reorder_probs(p_model):
        p_new = np.zeros_like(p_model)
        for our in range(NUM_C):
            model_idx = mapping.get(our, our)  # identity if missing
            p_new[our] = p_model[model_idx]
        return p_new

    return mapping, reorder_probs

perm_resnet_map, resnet_reorder = infer_perm_on_subset("resnet50")
perm_vit_map,    vit_reorder    = infer_perm_on_subset("vit")
print("ResNet-50 index mapping (our_idx -> resnet_idx):", perm_resnet_map)
print("ViT      index mapping (our_idx -> vit_idx):   ", perm_vit_map)

# Aligned prediction for all models 
@torch.no_grad()
def predict_proba_all_models_aligned(pil_img: Image.Image) -> dict:
    outs = {}

    x = preprocess_224_05(pil_img).unsqueeze(0).to(device)
    outs["cnn"] = torch.softmax(cnn_model(x), dim=1).detach().cpu().numpy()[0]

    x = preprocess_resnet(pil_img).unsqueeze(0).to(device)
    p_res = torch.softmax(resnet_model(x), dim=1).detach().cpu().numpy()[0]
    outs["resnet50"] = resnet_reorder(p_res)

    x = preprocess_224_05(pil_img).unsqueeze(0).to(device)
    outs["densenet121"] = torch.softmax(densenet_model(x), dim=1).detach().cpu().numpy()[0]

    x = preprocess_effnet(pil_img).unsqueeze(0).to(device)
    outs["efficientnet_b3"] = torch.softmax(effnet_model(x), dim=1).detach().cpu().numpy()[0]

    x = preprocess_224_05(pil_img).unsqueeze(0).to(device)
    outs["convnext_tiny"] = torch.softmax(convnext_model(x), dim=1).detach().cpu().numpy()[0]

    x = preprocess_224_05(pil_img).unsqueeze(0).to(device)
    outs["mobilenet_v3"] = torch.softmax(mobilenet_model(x), dim=1).detach().cpu().numpy()[0]

    x = preprocess_vit(pil_img).unsqueeze(0).to(device)
    p_v = torch.softmax(vit_model(x).logits, dim=1).detach().cpu().numpy()[0]
    outs["vit"] = vit_reorder(p_v)

    stack = np.stack([outs[k] for k in outs.keys()], axis=0)
    outs["ensemble_mean"] = stack.mean(axis=0)
    return outs

# Score full test set 
model_keys = ["cnn","resnet50","densenet121","efficientnet_b3","convnext_tiny","mobilenet_v3","vit"]
labels = []
per_model_probs = {k: [] for k in model_keys}
ensemble_probs  = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Scoring (FULL test)"):
    p = row["path"]
    try:
        img = Image.open(p).convert("RGB")
    except:
        continue
    outs = predict_proba_all_models_aligned(img)
    for k in model_keys:
        per_model_probs[k].append(outs[k])
    ensemble_probs.append(outs["ensemble_mean"])
    labels.append(int(row["label"]))

labels = np.array(labels)
for k in model_keys:
    per_model_probs[k] = np.array(per_model_probs[k])
ensemble_probs = np.array(ensemble_probs)
print(f"Scored {len(labels)} images.")

# Build comparison table 
def metrics_from_probs(y_true, p):
    preds = p.argmax(axis=1)
    acc = accuracy_score(y_true, preds)
    f1  = f1_score(y_true, preds, average="weighted")
    try:
        aucM = roc_auc_score(y_true, p, multi_class="ovr", average="macro")
        aucm = roc_auc_score(y_true, p, multi_class="ovr", average="micro")
    except ValueError:
        aucM, aucm = np.nan, np.nan
    return acc, f1, aucM, aucm

def count_params(m): 
    return sum(p.numel() for p in m.parameters())/1e6

param_count = {
    "cnn":            count_params(cnn_model)            if "cnn_model" in globals() else np.nan,
    "resnet50":       count_params(resnet_model)         if "resnet_model" in globals() else np.nan,
    "densenet121":    count_params(densenet_model)       if "densenet_model" in globals() else np.nan,
    "efficientnet_b3":count_params(effnet_model)         if "effnet_model" in globals() else np.nan,
    "convnext_tiny":  count_params(convnext_model)       if "convnext_model" in globals() else np.nan,
    "mobilenet_v3":   count_params(mobilenet_model)      if "mobilenet_model" in globals() else np.nan,
    "vit":            count_params(vit_model)            if "vit_model" in globals() else np.nan,
    "ensemble_mean":  np.nan
}

rows = []
for k in model_keys:
    acc, f1, aucM, aucm = metrics_from_probs(labels, per_model_probs[k])
    rows.append({
        "Model": k,
        "Params (M)": round(param_count.get(k, np.nan), 3),
        "Accuracy": round(acc, 4),
        "F1 (weighted)": round(f1, 4),
        "AUC (macro, OvR)": round(aucM, 4) if not np.isnan(aucM) else np.nan,
        "AUC (micro, OvR)": round(aucm, 4) if not np.isnan(aucm) else np.nan
    })

acc_e, f1_e, aucM_e, aucm_e = metrics_from_probs(labels, ensemble_probs)
rows.append({
    "Model": "ensemble_mean",
    "Params (M)": param_count["ensemble_mean"],
    "Accuracy": round(acc_e, 4),
    "F1 (weighted)": round(f1_e, 4),
    "AUC (macro, OvR)": round(aucM_e, 4) if not np.isnan(aucM_e) else np.nan,
    "AUC (micro, OvR)": round(aucm_e, 4) if not np.isnan(aucm_e) else np.nan
})

comparison_df = pd.DataFrame(rows).sort_values(by=["Accuracy","F1 (weighted)"], ascending=False).reset_index(drop=True)
display(comparison_df)

# Save
out_csv = "/kaggle/working/model_comparison_full.csv"
comparison_df.to_csv(out_csv, index=False)
print(f"Saved comparison: {out_csv}")

# Extra: classification report + confusion matrix for ensemble
ensemble_preds = ensemble_probs.argmax(axis=1)
print("\nEnsemble — Classification Report (FULL test):\n")
print(classification_report(labels, ensemble_preds, target_names=class_names))

cm = confusion_matrix(labels, ensemble_preds)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Confusion Matrix — Ensemble (FULL test)")
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix_ensemble_full.png", dpi=300)
plt.show()
print("Saved: /kaggle/working/confusion_matrix_ensemble_full.png")

**Figure 3.** Confusion matrix of ensemble predictions on the HAM10000 test set, illustrating per-class classification performance.

### Ensemble — ROC‑AUC Curves

In [ ]:
plot_multiclass_roc(
    all_labels=labels,
    all_probs=ensemble_probs,
    class_names=class_names,
    title="ROC Curves — Ensemble (Mean)",
    outfile="/kaggle/working/roc_curves_ensemble.png"
)

**Figure 4.** ROC curves for the ensemble across all lesion classes, with macro and micro AUC values reported.

### Ensemble — Simple Weight Search on Validation

In [ ]:
# Load validation DataFrame
val_csv_path = "/kaggle/input/ham10000-balanced-split/val_balanced.csv"
assert os.path.exists(val_csv_path), f"Validation CSV not found at {val_csv_path}"
val_df = pd.read_csv(val_csv_path)

# Define subset of strong models
subset_models = ["resnet50", "vit", "efficientnet_b3"]

# Helper function to extract model probabilities on the DataFrame
def probs_from_models_on_df(df_in):
    labels = []
    probs = {k: [] for k in subset_models}
    for i in tqdm(range(len(df_in)), desc="Collecting per-model probs (val)"):
        row = df_in.iloc[i]
        label = int(row["label"])

        # Resolve image path
        if "image_path" in row and pd.notna(row["image_path"]):
            path = row["image_path"]
        elif "path" in row and pd.notna(row["path"]):
            path = row["path"]
        else:
            path = find_image_path_from_id(row["image_id"])  # fallback if necessary

        if path is None or not os.path.exists(path):
            continue

        # Load image
        pil_img = Image.open(path).convert("RGB")
        outs = predict_proba_all_models_aligned(pil_img)   

        for k in subset_models:
            probs[k].append(outs[k])
        labels.append(label)

    labels = np.array(labels)
    for k in probs:
        probs[k] = np.array(probs[k])
    return labels, probs

# Get validation labels and probabilities
val_labels_ws, val_probs_ws = probs_from_models_on_df(val_df)

# Evaluation function for custom weights
def evaluate_weights(weights_dict, probs_dict, labels):
    total = None
    for k, w in weights_dict.items():
        total = probs_dict[k] * w if total is None else total + probs_dict[k] * w
    preds = total.argmax(axis=1)
    acc = (preds == labels).mean()
    f1 = f1_score(labels, preds, average="weighted")
    return acc, f1, total

# Grid search for best weights
best = {"acc": 0.0, "f1": 0.0, "weights": None, "probs": None}
grid = np.linspace(0, 1, 11)

for w1 in grid:
    for w2 in grid:
        w3 = 1.0 - w1 - w2
        if w3 < 0 or w3 > 1:
            continue
        W = {subset_models[0]: w1, subset_models[1]: w2, subset_models[2]: w3}
        acc, f1, total = evaluate_weights(W, val_probs_ws, val_labels_ws)
        if acc > best["acc"]:
            best = {"acc": acc, "f1": f1, "weights": W, "probs": total}

# Print best result
print("\nBest validation weights (coarse grid):", best["weights"])
print(f"Val Acc: {best['acc']:.4f} | Val F1: {best['f1']:.4f}")

## 4.2 Statistical Analysis

### Prep: Per-Model Predictions & a Helper to Compute Metrics

In [ ]:
# Prep: predictions + metric helper 
assert isinstance(labels, np.ndarray) and labels.ndim == 1, "labels must be 1D np.array"
assert isinstance(per_model_probs, dict) and len(per_model_probs) > 0, "per_model_probs missing"
assert isinstance(ensemble_probs, np.ndarray) and ensemble_probs.ndim == 2, "ensemble_probs missing"

model_list = list(per_model_probs.keys())  # e.g., ["cnn","resnet50",...,"vit"]
all_probs = {**per_model_probs, "ensemble_mean": ensemble_probs}

# Predicted labels (argmax)
all_preds = {m: p.argmax(axis=1) for m, p in all_probs.items()}

def compute_metrics(y_true, probs):
    """Return Accuracy, Weighted F1, AUC macro (OvR), AUC micro (OvR)."""
    y_pred = probs.argmax(axis=1)
    acc = accuracy_score(y_true, y_pred)
    f1w = f1_score(y_true, y_pred, average="weighted")
    try:
        auc_macro = roc_auc_score(y_true, probs, multi_class="ovr", average="macro")
        auc_micro = roc_auc_score(y_true, probs, multi_class="ovr", average="micro")
    except ValueError:
        auc_macro, auc_micro = np.nan, np.nan
    return acc, f1w, auc_macro, auc_micro

### Bootstrap 95% CIs

In [ ]:
# Bootstrap CIs
rng = np.random.default_rng(42)

def bootstrap_metric_ci(y, probs, metric_fn, B=1000, alpha=0.05):
    """Generic bootstrap CI for a scalar metric computed from (y, probs)."""
    n = len(y)
    stats = []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        s = metric_fn(y[idx], probs[idx])
        stats.append(s)
    stats = np.array(stats)
    lo = np.nanpercentile(stats, 100*alpha/2)
    hi = np.nanpercentile(stats, 100*(1-alpha/2))
    return float(np.nanmean(stats)), float(lo), float(hi)

def metric_bundle(y, probs, B=1000):
    # define metric lambdas that accept (y, probs)
    m_acc = lambda yy, pp: accuracy_score(yy, pp.argmax(axis=1))
    m_f1w = lambda yy, pp: f1_score(yy, pp.argmax(axis=1), average="weighted")
    def m_auc_macro(yy, pp):
        try: return roc_auc_score(yy, pp, multi_class="ovr", average="macro")
        except: return np.nan
    def m_auc_micro(yy, pp):
        try: return roc_auc_score(yy, pp, multi_class="ovr", average="micro")
        except: return np.nan
    # point estimates + CIs
    acc, acc_lo, acc_hi = bootstrap_metric_ci(y, probs, m_acc, B=B)
    f1w, f1_lo, f1_hi  = bootstrap_metric_ci(y, probs, m_f1w, B=B)
    aucM, M_lo, M_hi   = bootstrap_metric_ci(y, probs, m_auc_macro, B=B)
    aucm, m_lo, m_hi   = bootstrap_metric_ci(y, probs, m_auc_micro, B=B)
    return {
        "Accuracy": (acc, acc_lo, acc_hi),
        "F1 (weighted)": (f1w, f1_lo, f1_hi),
        "AUC (macro, OvR)": (aucM, M_lo, M_hi),
        "AUC (micro, OvR)": (aucm, m_lo, m_hi)
    }

rows = []
for m in model_list + ["ensemble_mean"]:
    bundle = metric_bundle(labels, all_probs[m], B=1000)
    rows.append({
        "Model": m,
        "Accuracy": f"{bundle['Accuracy'][0]:.4f} [{bundle['Accuracy'][1]:.4f}, {bundle['Accuracy'][2]:.4f}]",
        "F1 (weighted)": f"{bundle['F1 (weighted)'][0]:.4f} [{bundle['F1 (weighted)'][1]:.4f}, {bundle['F1 (weighted)'][2]:.4f}]",
        "AUC (macro, OvR)": f"{bundle['AUC (macro, OvR)'][0]:.4f} [{bundle['AUC (macro, OvR)'][1]:.4f}, {bundle['AUC (macro, OvR)'][2]:.4f}]",
        "AUC (micro, OvR)": f"{bundle['AUC (micro, OvR)'][0]:.4f} [{bundle['AUC (micro, OvR)'][1]:.4f}, {bundle['AUC (micro, OvR)'][2]:.4f}]",
    })

ci_table = pd.DataFrame(rows)
display(ci_table)
ci_table.to_csv("/kaggle/working/metrics_with_95CI.csv", index=False)
print("Saved: /kaggle/working/metrics_with_95CI.csv")

**Table 5.** Reliability diagrams (Expected Calibration Error, ECE) showing calibration of ensemble predictions versus ground-truth probabilities.

### McNemar’s Test

In [ ]:
# McNemar tests: ensemble vs each model (paired accuracy) 
# Requires statsmodels (available on Kaggle). If missing:

ens_pred = all_preds["ensemble_mean"]
mcnemar_rows = []
for m in model_list:
    p = all_preds[m]
    # contingency table:
    # b = ens correct, model wrong
    # c = ens wrong,   model correct
    ens_correct = (ens_pred == labels)
    mod_correct = (p == labels)
    b = int(np.sum(ens_correct & ~mod_correct))
    c = int(np.sum(~ens_correct & mod_correct))
    table = [[0, b],
             [c, 0]]
    # exact=False with continuity correction is standard; exact=True for small b+c
    res = mcnemar(table=table, exact=(b+c)<25, correction=True)
    mcnemar_rows.append({
        "Comparison": f"Ensemble vs {m}",
        "b (Ens correct, Model wrong)": b,
        "c (Ens wrong, Model correct)": c,
        "McNemar p-value": f"{res.pvalue:.4g}"
    })

mcnemar_df = pd.DataFrame(mcnemar_rows)
display(mcnemar_df)
mcnemar_df.to_csv("/kaggle/working/mcnemar_ensemble_vs_models.csv", index=False)
print("Saved: /kaggle/working/mcnemar_ensemble_vs_models.csv")

**Table 3.** McNemar’s test results comparing the ensemble against each individual model on the HAM10000 test set. Reported are disagreement counts (b, c) and p-values.

### Δ-Metrics (Ensemble − Model)

In [ ]:
# Paired bootstrap for differences (Ensemble - Model) 
def paired_bootstrap_diff(y, probs_A, probs_B, metric_fn, B=1000):
    n = len(y); diffs = []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        a = metric_fn(y[idx], probs_A[idx])
        b = metric_fn(y[idx], probs_B[idx])
        diffs.append(a - b)
    diffs = np.array(diffs)
    est = float(np.mean(diffs))
    lo  = float(np.percentile(diffs, 2.5))
    hi  = float(np.percentile(diffs, 97.5))
    return est, lo, hi

# define metrics
m_acc = lambda yy, pp: accuracy_score(yy, pp.argmax(axis=1))
m_f1w = lambda yy, pp: f1_score(yy, pp.argmax(axis=1), average="weighted")
def m_auc_macro(yy, pp):
    try: return roc_auc_score(yy, pp, multi_class="ovr", average="macro")
    except: return np.nan

paired_rows = []
for m in model_list:
    for name, fn in [("ΔAccuracy", m_acc), ("ΔF1 (weighted)", m_f1w), ("ΔAUC (macro, OvR)", m_auc_macro)]:
        est, lo, hi = paired_bootstrap_diff(labels, all_probs["ensemble_mean"], all_probs[m], fn, B=1000)
        paired_rows.append({
            "Comparison": f"Ensemble - {m}",
            "Metric Δ": name,
            "Estimate": f"{est:.4f}",
            "95% CI": f"[{lo:.4f}, {hi:.4f}]"
        })

paired_df = pd.DataFrame(paired_rows)
display(paired_df)
paired_df.to_csv("/kaggle/working/paired_bootstrap_differences.csv", index=False)
print("Saved: /kaggle/working/paired_bootstrap_differences.csv")

**Table 4.** Performance deltas (Ensemble – Individual model) with 95% confidence intervals for Accuracy, Weighted F1, and ROC–AUC (macro) on the HAM10000 test set.

### (Optional) Calibration: ECE + Reliability Curves (Multiclass)

In [ ]:
# Optional: Calibration (ECE) 
def multiclass_ece(y_true, probs, n_bins=15):
    # Use max-confidence binning (common in multiclass ECE)
    conf = probs.max(axis=1)
    preds = probs.argmax(axis=1)
    correct = (preds == y_true).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        mask = (conf > lo) & (conf <= hi)
        if not np.any(mask): 
            continue
        acc_bin = correct[mask].mean()
        conf_bin = conf[mask].mean()
        ece += (mask.mean()) * abs(acc_bin - conf_bin)
    return float(ece)

cal_rows = []
for m in model_list + ["ensemble_mean"]:
    ece = multiclass_ece(labels, all_probs[m], n_bins=15)
    cal_rows.append({"Model": m, "ECE (↓ better)": round(ece, 4)})
cal_df = pd.DataFrame(cal_rows).sort_values("ECE (↓ better)")
display(cal_df)
cal_df.to_csv("/kaggle/working/calibration_ece.csv", index=False)
print("Saved: /kaggle/working/calibration_ece.csv")

### Master Comparison Table

In [ ]:
# Master table combining point estimates + CIs 
def point_and_ci_str(y, probs, B=1000):
    mb = metric_bundle(y, probs, B=B)
    out = {
        "Accuracy": f"{mb['Accuracy'][0]:.4f} [{mb['Accuracy'][1]:.4f}, {mb['Accuracy'][2]:.4f}]",
        "F1 (weighted)": f"{mb['F1 (weighted)'][0]:.4f} [{mb['F1 (weighted)'][1]:.4f}, {mb['F1 (weighted)'][2]:.4f}]",
        "AUC (macro, OvR)": f"{mb['AUC (macro, OvR)'][0]:.4f} [{mb['AUC (macro, OvR)'][1]:.4f}, {mb['AUC (macro, OvR)'][2]:.4f}]",
        "AUC (micro, OvR)": f"{mb['AUC (micro, OvR)'][0]:.4f} [{mb['AUC (micro, OvR)'][1]:.4f}, {mb['AUC (micro, OvR)'][2]:.4f}]",
    }
    return out

master_rows = []
for m in model_list + ["ensemble_mean"]:
    row = {"Model": m}
    row.update(point_and_ci_str(labels, all_probs[m], B=1000))
    master_rows.append(row)

master_df = pd.DataFrame(master_rows)
display(master_df)
master_df.to_csv("/kaggle/working/master_results_with_95CI.csv", index=False)
print("Saved: /kaggle/working/master_results_with_95CI.csv")

### LaTeX Export of Results

In [ ]:
# Export Master Results to LaTeX 
latex_table = master_df.rename(columns={
    "Model": "Model",
    "Accuracy": "Accuracy (95% CI)",
    "F1 (weighted)": "F1 (weighted, 95% CI)",
    "AUC (macro, OvR)": "AUC (macro, OvR, 95% CI)",
    "AUC (micro, OvR)": "AUC (micro, OvR, 95% CI)"
}).to_latex(
    index=False,
    escape=False,        # keep brackets []
    column_format="lcccc",  # left align model, center metrics
    caption="Performance comparison of individual models and ensemble on HAM10000 (95% bootstrap confidence intervals).",
    label="tab:results"
)

# Save to file
with open("/kaggle/working/master_results_with_95CI.tex", "w") as f:
    f.write(latex_table)

print("Saved: /kaggle/working/master_results_with_95CI.tex")
print("\nPreview:\n")
print(latex_table)

### Statistical Significance & Performance Δ-Metrics (Ensemble vs Individual Models)

In [ ]:
# Statistical Significance & Δ-Metrics (LaTeX-ready)

# Assertions & prep
assert isinstance(labels, np.ndarray) and labels.ndim == 1, "labels must be 1D np.ndarray"
assert isinstance(per_model_probs, dict) and len(per_model_probs) > 0, "per_model_probs dict missing"
assert isinstance(ensemble_probs, np.ndarray) and ensemble_probs.ndim == 2, "ensemble_probs missing"
assert 'Model' in comparison_df.columns, "comparison_df must be built first"

N = len(labels)
model_keys = list(per_model_probs.keys())
for k in model_keys:
    assert per_model_probs[k].shape[0] == N, f"Length mismatch for {k}"
assert ensemble_probs.shape[0] == N, "Length mismatch for ensemble_probs"

# Pretty names (for tables)
pretty = {
    "cnn": "CNN",
    "resnet50": "ResNet-50",
    "densenet121": "DenseNet-121",
    "efficientnet_b3": "EfficientNet-B3",
    "convnext_tiny": "ConvNeXt-Tiny",
    "mobilenet_v3": "MobileNetV3",
    "vit": "ViT",
    "ensemble_mean": "Ensemble",
}

# Convert probs -> hard predictions
def to_preds(prob_mat: np.ndarray) -> np.ndarray:
    return prob_mat.argmax(axis=1)

ensemble_pred = to_preds(ensemble_probs)
indiv_preds   = {k: to_preds(per_model_probs[k]) for k in model_keys}

# McNemar test (Ensemble vs Each Model) 
def mcnemar_p(y_true, y_pred_a, y_pred_b):
    """
    McNemar's exact binomial test for paired classifiers.
    b = A wrong & B correct, c = A correct & B wrong
    p-value is two-sided binom test on min(b,c) with n=b+c, p=0.5
    """
    a_correct = (y_pred_a == y_true)
    b_correct = (y_pred_b == y_true)
    b = np.sum(~a_correct &  b_correct)  # A wrong,  B right
    c = np.sum( a_correct & ~b_correct)  # A right,  B wrong
    n = b + c
    if n == 0:
        # Models make identical decisions for all samples; define p=1.0
        return 1.0, b, c, n
    p = binomtest(min(b, c), n=n, p=0.5, alternative='two-sided').pvalue
    return float(p), int(b), int(c), int(n)

# Build McNemar rows (Ensemble vs each)
mcnemar_rows = []
for k in model_keys:
    p, b, c, n = mcnemar_p(labels, indiv_preds[k], ensemble_pred)
    # significance stars
    if   p < 0.001: stars = "***"
    elif p < 0.01:  stars = "**"
    elif p < 0.05:  stars = "*"
    else:           stars = ""
    mcnemar_rows.append({
        "Model": pretty.get(k, k),
        "b (Model wrong, Ensemble right)": b,
        "c (Model right, Ensemble wrong)": c,
        "Disagreements (b+c)": n,
        "p-value": f"{p:.4g}",
        "Sig.": stars
    })

mcnemar_df = pd.DataFrame(mcnemar_rows).sort_values(by="Model").reset_index(drop=True)

# Export LaTeX (McNemar table: Ensemble vs models)
mcnemar_latex = mcnemar_df.to_latex(
    index=False,
    escape=False,
    column_format="lrrrrc",
    caption="McNemar’s test between the Ensemble and each individual model on the HAM10000 test set.",
    label="tab:mcnemar_ensemble_vs_models"
)
with open("/kaggle/working/mcnemar_ensemble_vs_models.tex", "w") as f:
    f.write(mcnemar_latex)

print("Saved: /kaggle/working/mcnemar_ensemble_vs_models.tex")
print("\n[Preview] McNemar — Ensemble vs Models:\n")
print(mcnemar_df.head(10).to_string(index=False))

# Δ-Metrics (Ensemble − Model)
# Make sure comparison_df has Ensemble row called 'ensemble_mean'
assert (comparison_df['Model'] == 'ensemble_mean').any(), "comparison_df must include 'ensemble_mean'"

# Extract ensemble metrics
row_ens = comparison_df.set_index('Model').loc['ensemble_mean']

def safe_delta(ens, val):
    if pd.isna(ens) or pd.isna(val):
        return np.nan
    return float(ens) - float(val)

delta_rows = []
for _, r in comparison_df.iterrows():
    m = r['Model']
    if m == 'ensemble_mean':
        continue
    delta_rows.append({
        "Model": pretty.get(m, m),
        "Δ Accuracy":        f"{safe_delta(row_ens['Accuracy'],         r['Accuracy']):+.4f}",
        "Δ F1 (weighted)":   f"{safe_delta(row_ens['F1 (weighted)'],    r['F1 (weighted)']):+.4f}",
        "Δ AUC (macro, OvR)":f"{safe_delta(row_ens['AUC (macro, OvR)'], r['AUC (macro, OvR)']):+.4f}",
        "Δ AUC (micro, OvR)":f"{safe_delta(row_ens['AUC (micro, OvR)'], r['AUC (micro, OvR)']):+.4f}",
    })
delta_df = pd.DataFrame(delta_rows).sort_values(by="Model").reset_index(drop=True)

# Export LaTeX (Δ table)
delta_latex = delta_df.to_latex(
    index=False,
    escape=False,
    column_format="lcccc",
    caption="Performance deltas (Ensemble minus each individual model) on the HAM10000 test set.",
    label="tab:delta_ensemble_vs_models"
)
with open("/kaggle/working/delta_ensemble_vs_models.tex", "w") as f:
    f.write(delta_latex)

print("\nSaved: /kaggle/working/delta_ensemble_vs_models.tex")
print("\n[Preview] Δ-Metrics (Ensemble − Model):\n")
print(delta_df.head(10).to_string(index=False))

# 5. Clinical Relevance

Accurate classification of skin lesions across heterogeneous datasets is crucial for deploying trustworthy AI-assisted diagnostic systems in clinical practice. This study demonstrates that ensemble-based deep learning models significantly improve robustness and consistency, particularly under domain shift scenarios such as external validation on the ISIC 2019 dataset.

The proposed framework addresses key clinical requirements:
- **High Diagnostic Accuracy:** Ensemble predictions achieve superior performance across internal and external datasets.
- **Interpretability:** Grad-CAM visualizations provide per-class heatmaps that help clinicians understand model focus areas and support visual trust.
- **Reliability:** Expected Calibration Error (ECE) analysis quantifies the alignment between predicted confidence and true accuracy—critical for clinical decision-making under uncertainty.
- **Reproducibility:** Exported results—including confusion matrices, ROC curves, classification reports, and LaTeX/CSV-formatted metrics—facilitate validation and integration into medical research workflows.

While this study did not involve direct collaboration with dermatologists, the framework is designed with clinical deployment in mind. It can support practitioners in triaging high-risk cases, especially in low-resource or remote healthcare settings where access to expert dermatologists is limited.

Future work should include expert-in-the-loop validation to enhance usability, safety, and trust in real-world dermatology applications.

# 6. Discussion

Our results reveal substantial variation in cross-domain generalization across model architectures. While DenseNet-121, EfficientNet-B3, and ConvNeXt-Tiny maintained strong performance when evaluated on the external ISIC 2019 dataset, ResNet-50 and Vision Transformer (ViT) exhibited notable declines in ROC–AUC and F1-score, indicating reduced robustness under domain shift.

Despite ViT’s theoretical strength in capturing global image context, its performance lagged in the external evaluation. Several factors likely contributed to this outcome:
- **Overfitting to HAM10000 distribution**, especially given ViT’s sensitivity to dataset size and lack of inductive biases.
- **Feature fragility under domain shift**, where learned representations may not transfer well to unseen data.
- **Class imbalance**, which disproportionately impacts minority classes in the absence of specialized loss handling.

In contrast, the ensemble strategy—combining predictions from multiple models—consistently delivered improved performance. By averaging softmax probabilities, the ensemble achieved higher accuracy and AUC on both internal and external datasets, aligning with recent findings by Thwin and Park (2024) and Efat et al. (2024). These studies showed that ensembles not only enhance classification but also improve interpretability when paired with methods like Grad-CAM.

Our study additionally incorporated Expected Calibration Error (ECE) analysis, providing insights into prediction reliability. While the ensemble model excelled in classification metrics, its calibration was slightly worse than models like ViT and MobileNetV3, suggesting a trade-off between accuracy and confidence reliability. This trade-off warrants further investigation, especially for clinical applications where calibrated probabilities are critical.

However, the current pipeline has limitations:
- **Interpretability on external data remains limited**, as Grad-CAM visualizations were generated only for HAM10000.
- **Lack of dermatologist collaboration** prevents assessment of clinical relevance and real-world utility of model outputs.

Overall, this work demonstrates the practical benefits of ensemble learning for skin lesion diagnosis and lays the groundwork for future studies integrating expert-in-the-loop validation, domain adaptation, and advanced explainability across datasets.

# 7. Conclusion

This study conducted a comprehensive evaluation of seven deep learning models—CNN, ResNet-50, DenseNet-121, EfficientNet-B3, ConvNeXt-Tiny, MobileNetV3, and Vision Transformer (ViT)—for the task of automated skin lesion classification. All models were trained on the HAM10000 dataset and externally validated on the ISIC 2019 dataset to assess generalization under domain shift.

**Key takeaways:**
- **Ensemble learning** significantly improved classification performance across internal and external datasets, outperforming all individual models on accuracy, F1-score, and ROC–AUC.
- **DenseNet-121, ConvNeXt-Tiny, and EfficientNet-B3** emerged as the strongest standalone performers, showing robustness to domain variation.
- **ViT and ResNet-50** exhibited limited generalization, underscoring the importance of architecture-specific tuning or pretraining strategies for clinical imaging tasks.
- All evaluation outputs—including classification reports, ROC curves, confidence intervals, Grad-CAM visualizations, and LaTeX/CSV-formatted tables—are **fully exportable and publication-ready**.

This work contributes a reproducible benchmark and analysis pipeline for the development of clinically relevant, generalizable AI systems in dermatology. Future iterations will explore domain adaptation techniques, multimodal learning, and expert-in-the-loop validation to enhance deployment readiness in real-world healthcare environments.

# 8. Limitations and Future Work

Despite the promising results and strong ensemble performance, this study has several limitations that should be acknowledged:

- **Class Imbalance:** Both HAM10000 and ISIC 2019 exhibit significant class imbalance, which may have influenced model learning dynamics and contributed to reduced performance in minority classes.
- **Grad-CAM for External Data:** Visual interpretability through Grad-CAM was only applied to the internal HAM10000 test set. The absence of Grad-CAM analysis on ISIC 2019 limits transparency of model behavior under domain shift.
- **Model Generalization Gaps:** Architectures such as ViT and ResNet-50 underperformed during external validation, suggesting susceptibility to overfitting and potential misalignment between learned representations and unseen domains.
- **Lack of Expert Clinical Review:** No board-certified dermatologists were involved in assessing the models’ predictions. This limits immediate clinical translatability and underlines the need for expert-in-the-loop validation.

**Future directions** include:

- Implementing **domain adaptation techniques** (e.g., adversarial training, feature alignment) to improve robustness across datasets with differing characteristics.
- Developing **multi-modal architectures** that incorporate patient metadata (age, sex, lesion location) alongside images to enhance diagnostic performance.
- Extending the pipeline to support **mobile or edge deployment**, enabling real-time inference in low-resource settings or teledermatology workflows.
- Collaborating with clinicians to conduct **qualitative validation**, including human-AI comparison studies and usability assessments in realistic diagnostic scenarios.

Addressing these aspects in future work will be essential for translating high-performing AI models into trustworthy, interpretable, and deployable clinical tools.